In [1]:
!pip uninstall -y -q torchvision torchaudio torchao bitsandbytes auto-gptq optimum

In [2]:
!pip install -q --no-deps \
    "transformers==4.46.3" \
    "huggingface_hub==0.26.2" \
    "sentence-transformers==3.3.1" \
    "peft==0.13.2" \
    "tokenizers==0.20.3"
!pip install -q --no-deps "safetensors" "regex" "filelock" "fsspec"

print("\n" + "=" * 70)
print("INSTALL DONE.  Now: Run > Restart kernel, then run from the top.")
print("=" * 70)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 71.6 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.5/447.5 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.8/268.8 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 84.7 MB/s eta 0:00:00:00:01

INSTALL DONE.  Now: Run > Restart kernel, then run from the top.


In [3]:
import numpy, scipy, sklearn, transformers, huggingface_hub, sentence_transformers, peft
print(f"numpy {numpy.__version__} | scipy {scipy.__version__} | sklearn {sklearn.__version__}")
print(f"transformers {transformers.__version__} (want 4.46.3)")
print(f"huggingface_hub {huggingface_hub.__version__} (want 0.26.2)")
print(f"sentence_transformers {sentence_transformers.__version__} (want 3.3.1)")
from scipy.stats import spearmanr
from sklearn.metrics import f1_score
assert transformers.__version__.startswith("4."), (
    f"transformers is {transformers.__version__}; re-run the install cell AND RESTART."
)
print("Environment OK.")


numpy 2.0.2 | scipy 1.16.3 | sklearn 1.6.1
transformers 4.46.3 (want 4.46.3)
huggingface_hub 0.26.2 (want 0.26.2)
sentence_transformers 3.3.1 (want 3.3.1)
Environment OK.


In [4]:
import gc
import json
import math
import random
from contextlib import nullcontext
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import load_dataset
from scipy.stats import spearmanr
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

from sentence_transformers import SentenceTransformer
from sentence_transformers.util import batch_to_device
from peft import (
    LoraConfig,
    TaskType,
    PeftModel,
    PeftModelForFeatureExtraction,
    get_peft_model_state_dict,
    set_peft_model_state_dict,
)

import nltk
for _pkg in ("wordnet", "omw-1.4", "punkt", "punkt_tab",
             "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"):
    print(f"NLTK: fetching {_pkg}...", end=" ", flush=True)
    try:
        nltk.download(_pkg, quiet=True)
        print("done")
    except Exception as _e:
        print(f"skipped ({str(_e)[:40]})")
from nltk.corpus import wordnet

SEED = 42

def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)
print("PyTorch:", torch.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

NLTK: fetching wordnet... done
NLTK: fetching omw-1.4... done
NLTK: fetching punkt... done
NLTK: fetching punkt_tab... done
NLTK: fetching averaged_perceptron_tagger... done
NLTK: fetching averaged_perceptron_tagger_eng... done
Device: cuda
PyTorch: 2.10.0+cu128
GPU: Tesla T4


In [5]:
# Base encoder for the LoRA alignment run: MPNet-base (all-mpnet-base-v2).
# Unlike supervised SimCSE (which needs its trained MLP pooler head over the
# [CLS] token, i.e. `pooler_output`), MPNet's sentence embedding is the
# standard sentence-transformers MEAN-POOLED token embeddings -- the pooling
# strategy this checkpoint was actually trained with. build_base_model()
# below builds the model with a plain Transformer + mean-pooling Pooling
# module (sentence_transformers.models), so training and evaluation use
# exactly the pooling MPNet expects, with no custom pooler-head code needed.
BASE_MODEL = "sentence-transformers/all-mpnet-base-v2"

from transformers import AutoConfig
# MPNet-base's native context length -- read from the checkpoint's own
# config instead of hardcoding, so this stays correct if BASE_MODEL is
# swapped again later. (Note: all-mpnet-base-v2's sentence-transformers
# config caps effective usage at 384 tokens even though the underlying
# HF config allows more; CFG.max_seq_length below is what actually governs
# training/eval truncation.)
BASE_MODEL_MAX_SEQ_LEN = min(
    getattr(AutoConfig.from_pretrained(BASE_MODEL), "max_position_embeddings", 512),
    512,
)

TRAIN_PATH = "/kaggle/input/datasets/aref111n/embedding/final_train.csv"
DEV_PATH = "/kaggle/input/datasets/aref111n/embedding/final_dev.csv"
TEST_PATH = "/kaggle/input/datasets/aref111n/embedding/final_test.csv"

@dataclass
class TrainConfig:
    train_csv: str = TRAIN_PATH
    dev_csv: str = DEV_PATH
    test_csv: str = TEST_PATH
    output_dir: str = "/kaggle/working/align_mpnet_mtl"

    max_seq_length: int = 128

    use_task_train_data: bool = True
    use_generic_augmentation: bool = True
    augmented_anchor_count: int = 8000
    qqp_train_pairs: int = 60000
    paws_train_pairs: int = 45000
    nli_triplet_count: int = 100000
    # 6000, not the full 10000 -- prior runs' C3 growth rate consistently
    # decelerated/plateaued in the 4500-6000 range, so this is enough steps
    # to see whether r=32 changes that trajectory shape, without paying for
    # the full remaining steps if it doesn't.
    max_steps: int = 12000
    eval_every: int = 500                                                      
                                                                            
    warmup_steps: int = 350                                           
    lr: float = 2e-5
    weight_decay: float = 0.01
    max_grad_norm: float = 1.0

    relation_batch_size: int = 12   
    pair_batch_size: int = 48       
    temperature: float = 0.05
    # rank_temperature and weight_rank REMOVED -- they only controlled the
    # within-anchor ranking loss in relation_step(), which has been
    # discarded (relation_step is now masked_infonce only).
    # rank_margin is KEPT: it's also used, separately, by the embedding-
    # geometry diagnostic (evaluate_embedding_geometry's
    # "equiv_beats_opposite" check) -- unrelated to the training loss.
    rank_margin: float = 0.10
    cosent_scale: float = 20.0
    # Lowered from r=32/alpha=64 -- that capacity, spread across every
    # attention projection AND the full FFN, gave the adapter enough room to
    # overwrite the base MPNet model's general-purpose structure in favor
    # of fitting the training dataset stew (QQP+PAWS+NLI+STS+criteria),
    # which is the likely cause of the MR/CR regression below baseline.
    # r=8/alpha=16 keeps the classic LoRA-paper capacity ratio (alpha=2r)
    # while cutting trainable parameters ~4x.
    # Back to r=16/alpha=32 -- capacity (r=32) and lora_targets (+key) are
    # dropped as variables for THIS run, so the jumbled-column regeneration
    # (full-permutation -> max-3-word-swap, much harder now) and the
    # task_weights rebalance get tested against the known r=16 baseline,
    # not stacked with capacity changes too.
    lora_r: int = 24
    lora_alpha: int = 48
    # Slightly higher dropout as extra regularization against re-overfitting
    # to the smaller-capacity adapter's now-narrower solution space.
    lora_dropout: float = 0.1
    # MPNet names its attention projections "q"/"k"/"v" with output
    # projection "o" (transformers.models.mpnet.MPNetSelfAttention) --
    # unlike BERT/SimCSE's "query"/"key"/"value" naming.
    # DROPPED the FFN block ("intermediate.dense"/"output.dense") from the
    # target list -- the FFN is where most of a transformer's general
    # semantic/world knowledge lives, and leaving it frozen keeps that
    # knowledge intact instead of letting it drift toward the training
    # dataset mix. Only "q"/"v" are tuned (the classic minimal LoRA
    # target set); "k" is dropped too since query already carries the
    # attention-pattern adaptation and key/query are redundant in what they
    # can express for this purpose.
    lora_targets: Tuple[str, ...] = (
        "q", "v"
    )
    encode_batch_size: int = 64
    alignment_min_gain: float = 0.03
    delta_tolerance: float = 0.002
    checkpoint_mrpc_pairs: int = 3000
    checkpoint_qqp_pairs: int = 4000                                                                                                                               
    checkpoint_paws_pairs: int = 3000
    final_qqp_tune_pairs: int = 10000
    final_qqp_eval_pairs: int = 5000                                                                                                                                                                                                                         
    final_paws_tune_pairs: int = 8000
    wise_ft_lambdas: Tuple[float, ...] = (0.3, 0.4, 0.5, 0.6, 0.75, 0.85, 1.0)
    wise_ft_criteria_weight: float = 0.3                                                  
    criteria_quad_count: int = 40000                                                                   
    criteria_eps_min: float = 0.05
    criteria_eps_max: float = 0.05                                     
    criteria_antonym_weight: float = 1.0
    criteria_jumble_weight: float = 0.5                                                                       
    criteria_eps: float = 0.05                                          
    criteria_probe_pairs: int = 1000                                                                                         
    # Per-eval rotating sample: each eval draws a fresh shuffle of this many
    # quads from the ~1000-anchor dev pool above, instead of reusing one
    # fixed set every time. 500-of-1000 leaves real slack (unlike 1000-of-
    # ~1000, which would just reshuffle order with no content variation).
    criteria_probe_eval_pairs: int = 500                     
    polarity_ceiling: float = 0.78
    nli_margin: float = 0.05
    nli_sep_weight: float = 0.20                                                                                              
    patience: Optional[int] = 3
    resume_checkpoint_every: int = 250                                     
    resume_checkpoint_filename: str = "resume_checkpoint.pt"                                                                                                                            
    allow_incompatible_resume: bool = False
    @property
    def task_weights(self) -> Dict[str, float]:                                 
        if self.use_task_train_data:
            # criteria doubled (0.10->0.20): C4 saturates ~90%+ on the same
            # pool C3 stalls on, so the bottleneck looks like gradient
            # competition, not data quantity -- give criteria more sampling
            # frequency. Trimmed from stsb/paws specifically (not evenly),
            # since those are the two objectives most tied to the earlier
            # MR/CR downstream regression.
            return {
                "relation": 0.15,
                "criteria": 0.20,
                "stsb": 0.20,
                "qqp": 0.10,
                "paws": 0.10,
                "nli": 0.25,
            }

        return {
            "relation": 0.40,
            "criteria": 0.15,
            "nli": 0.45,
        }

CFG = TrainConfig()
QUICK_TEST = False                                                

if QUICK_TEST:
    CFG.max_steps = 6000
    CFG.eval_every = 750
    CFG.warmup_steps = 80
    CFG.qqp_train_pairs = 8000
    CFG.paws_train_pairs = 8000
    CFG.nli_triplet_count = 15000
    CFG.augmented_anchor_count = 800
    CFG.checkpoint_qqp_pairs = 2000
    CFG.final_qqp_eval_pairs = 2000
    CFG.wise_ft_lambdas = (0.75, 1.0)
Path(CFG.output_dir).mkdir(parents=True, exist_ok=True)
print(json.dumps(asdict(CFG), indent=2))
print("task weights:", CFG.task_weights)

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

{
  "train_csv": "/kaggle/input/datasets/aref111n/embedding/final_train.csv",
  "dev_csv": "/kaggle/input/datasets/aref111n/embedding/final_dev.csv",
  "test_csv": "/kaggle/input/datasets/aref111n/embedding/final_test.csv",
  "output_dir": "/kaggle/working/align_mpnet_mtl",
  "max_seq_length": 128,
  "use_task_train_data": true,
  "use_generic_augmentation": true,
  "augmented_anchor_count": 8000,
  "qqp_train_pairs": 60000,
  "paws_train_pairs": 45000,
  "nli_triplet_count": 100000,
  "max_steps": 12000,
  "eval_every": 500,
  "warmup_steps": 350,
  "lr": 2e-05,
  "weight_decay": 0.01,
  "max_grad_norm": 1.0,
  "relation_batch_size": 12,
  "pair_batch_size": 48,
  "temperature": 0.05,
  "rank_margin": 0.1,
  "cosent_scale": 20.0,
  "lora_r": 24,
  "lora_alpha": 48,
  "lora_dropout": 0.1,
  "lora_targets": [
    "q",
    "v"
  ],
  "encode_batch_size": 64,
  "alignment_min_gain": 0.03,
  "delta_tolerance": 0.002,
  "checkpoint_mrpc_pairs": 3000,
  "checkpoint_qqp_pairs": 4000,
  "check

In [6]:
RELATION_ORDER = [
    "paraphrase", "synonym", "entailment",
    "contradiction", "antonym","negation",
    "jumbled",
    "distinction",
]                                                         
RELATION_RANK = {
    "paraphrase": 3,
    "synonym": 3,
    "entailment": 3,
    "contradiction": 2,
    "antonym": 2,
    "negation": 2,                                                              
    "jumbled": 1,
    "distinction": 0,
}

POSITIVE_TYPES = {"paraphrase", "synonym", "entailment"}                                          
HARD_NEGATIVE_TYPES = {"contradiction", "antonym", "negation", "jumbled", "distinction"}               

TARGET_EVAL_RELATIONS = [
    "paraphrase", "synonym", "entailment",
    "contradiction", "antonym", "negation",
    "jumbled", "distinction",
]

EXTRA_REPORT_RELATIONS: List[str] = []

HARD_NEGATIVE_MIX = {
    "antonym": 0.25,
    "contradiction": 0.25,
    "negation": 0.15,
    "jumbled": 0.20,
    "distinction": 0.15,
}

RELATION_ALIASES = {
    "syn": "synonym", "synonyms": "synonym",
    "ant": "antonym", "antonyms": "antonym",
    "para": "paraphrase", "paraphrasing": "paraphrase",
    "jumbling": "jumbled", "jumble": "jumbled",
    "shuffle": "jumbled", "shuffled": "jumbled",
    "distinct": "distinction", "difference": "distinction",
    "contradictory": "contradiction", "contradict": "contradiction",
    "entails": "entailment",
    "negated": "negation", "negate": "negation", "negations": "negation",
}

def normalize_text(text: Any) -> str:
    return "" if pd.isna(text) else " ".join(str(text).strip().lower().split())

def normalize_relation(value: Any) -> str:
    relation = normalize_text(value).replace("-", "_").replace(" ", "_")
    return RELATION_ALIASES.get(relation, relation)

def lexical_jaccard(left: str, right: str) -> float:
    left_tokens = set(normalize_text(left).split())
    right_tokens = set(normalize_text(right).split())
    if not left_tokens and not right_tokens:
        return 1.0
    if not left_tokens or not right_tokens:
        return 0.0
    return len(left_tokens & right_tokens) / len(left_tokens | right_tokens)

def load_csv(path: str, name: str) -> pd.DataFrame:
    dataframe = pd.read_csv(path)
    if "relation" in dataframe.columns and "relation_type" not in dataframe.columns:
        dataframe = dataframe.rename(columns={"relation": "relation_type"})
    required = {"anchor", "candidate", "relation_type"}
    missing = required - set(dataframe.columns)
    if missing:
        raise ValueError(f"{name} is missing required columns: {sorted(missing)}")

    dataframe = dataframe.copy()
    dataframe["anchor"] = dataframe["anchor"].astype(str).str.strip()
    dataframe["candidate"] = dataframe["candidate"].astype(str).str.strip()
    dataframe["relation_type"] = dataframe["relation_type"].map(normalize_relation)

    unsupported = dataframe.loc[
        ~dataframe["relation_type"].isin(RELATION_RANK), "relation_type"
    ]
    if len(unsupported):
        print(f"  !! {name}: UNRECOGNIZED relation values being dropped:")
        print(unsupported.value_counts().to_string())
        print("     (add them to RELATION_RANK or RELATION_ALIASES to keep them)")
    before = len(dataframe)
    dataframe = dataframe[dataframe["relation_type"].isin(RELATION_RANK)].copy()
    unsupported_removed = before - len(dataframe)

    dataframe = dataframe.drop_duplicates(
        subset=["anchor", "candidate", "relation_type"]
    ).reset_index(drop=True)

    same_text = (
        dataframe["anchor"].map(normalize_text)
        == dataframe["candidate"].map(normalize_text)
    )
    invalid_negative = same_text & dataframe["relation_type"].isin(HARD_NEGATIVE_TYPES)
    invalid_removed = int(invalid_negative.sum())
    dataframe = dataframe.loc[~invalid_negative].reset_index(drop=True)

    print(f"{name}: {len(dataframe):,} rows, {dataframe['anchor'].nunique():,} anchors")
    print("  unsupported rows removed:", unsupported_removed)
    print("  invalid negative self-pairs removed:", invalid_removed)
    return dataframe

                                                                             
                                                                        
                                                          
raw_train = load_csv(CFG.train_csv, "train")
raw_dev = load_csv(CFG.dev_csv, "dev")
raw_test = load_csv(CFG.test_csv, "test")

print("\ntrain relation counts:")
display(raw_train["relation_type"].value_counts())
print("dev relation counts:")
display(raw_dev["relation_type"].value_counts())
print("test relation counts:")
display(raw_test["relation_type"].value_counts())


  !! train: UNRECOGNIZED relation values being dropped:
relation_type
neutral    7999
     (add them to RELATION_RANK or RELATION_ALIASES to keep them)
train: 59,390 rows, 7,999 anchors
  unsupported rows removed: 7999
  invalid negative self-pairs removed: 0
  !! dev: UNRECOGNIZED relation values being dropped:
relation_type
neutral    1000
     (add them to RELATION_RANK or RELATION_ALIASES to keep them)
dev: 7,425 rows, 1,000 anchors
  unsupported rows removed: 1000
  invalid negative self-pairs removed: 0
  !! test: UNRECOGNIZED relation values being dropped:
relation_type
neutral    1001
     (add them to RELATION_RANK or RELATION_ALIASES to keep them)
test: 7,432 rows, 1,001 anchors
  unsupported rows removed: 1001
  invalid negative self-pairs removed: 0

train relation counts:


relation_type
paraphrase       7999
synonym          7999
antonym          7999
entailment       7999
contradiction    7999
jumbled          7999
distinction      7999
negation         3397
Name: count, dtype: int64

dev relation counts:


relation_type
paraphrase       1000
synonym          1000
antonym          1000
entailment       1000
contradiction    1000
jumbled          1000
distinction      1000
negation          425
Name: count, dtype: int64

test relation counts:


relation_type
paraphrase       1001
synonym          1001
antonym          1001
entailment       1001
contradiction    1001
jumbled          1001
distinction      1001
negation          425
Name: count, dtype: int64

In [7]:
train_df = raw_train.copy()
dev_df = raw_dev.copy()
test_df = raw_test.copy()

for name, split in [("train", train_df), ("dev", dev_df), ("test", test_df)]:
    print(f"{name}: {len(split):,} rows, {split['anchor'].nunique():,} anchors")

                                                                              
dev_anchor_set = set(dev_df["anchor"].map(normalize_text))
test_anchor_set = set(test_df["anchor"].map(normalize_text))
train_anchor_set = set(train_df["anchor"].map(normalize_text))

dev_test_overlap = dev_anchor_set & test_anchor_set
train_dev_overlap = train_anchor_set & dev_anchor_set
train_test_overlap = train_anchor_set & test_anchor_set
if dev_test_overlap or train_dev_overlap or train_test_overlap:
    print("\nnote: predefined partitions are used as-is (not modified); "
          "anchor overlap detected:")
    print(f"  dev/test overlap:   {len(dev_test_overlap):,} anchors")
    print(f"  train/dev overlap:  {len(train_dev_overlap):,} anchors")
    print(f"  train/test overlap: {len(train_test_overlap):,} anchors")
else:
    print("\nanchor-disjointness verified: train / dev / test share no anchors.")

print("\ndev relation coverage:")
display(dev_df["relation_type"].value_counts())
print("test relation coverage:")
display(test_df["relation_type"].value_counts())


train: 59,390 rows, 7,999 anchors
dev: 7,425 rows, 1,000 anchors
test: 7,432 rows, 1,001 anchors

anchor-disjointness verified: train / dev / test share no anchors.

dev relation coverage:


relation_type
paraphrase       1000
synonym          1000
antonym          1000
entailment       1000
contradiction    1000
jumbled          1000
distinction      1000
negation          425
Name: count, dtype: int64

test relation coverage:


relation_type
paraphrase       1001
synonym          1001
antonym          1001
entailment       1001
contradiction    1001
jumbled          1001
distinction      1001
negation          425
Name: count, dtype: int64

In [8]:
import os
from datasets import Dataset as HFDataset, DatasetDict

DATA_DIR = "/kaggle/input/datasets/aref111n/embedding"

def load_local_splits(name, files):
    """files: dict of split_name -> filename (parquet). Missing files are skipped."""
    print(f"[{name}] loading from local parquet...", flush=True)
    splits = {}
    for split, fname in files.items():
        path = os.path.join(DATA_DIR, fname)
        if not os.path.exists(path):
            print(f"  [{name}] WARNING: {path} not found, skipping split '{split}'")
            continue
        df = pd.read_parquet(path)
        splits[split] = HFDataset.from_pandas(df, preserve_index=False)
    ds = DatasetDict(splits)
    print(f"[{name}] loaded: " + ", ".join(f"{k}={len(v):,}" for k, v in ds.items()))
    return ds

                                                                           
                                                                       
                                                                            
                                                                      
                                                                           
stsb = load_local_splits("STS-B", {
    "train": "train-stsb.parquet",
    "validation": "validation-stsb.parquet",
    "test": "test-stsb.parquet",
})

def _normalize_stsb_label(ds):
    if "label" in ds.column_names:
        return ds
    for cand in ("score", "similarity_score", "similarity"):
        if cand in ds.column_names:
            return ds.rename_column(cand, "label")
    raise ValueError(f"STS-B: no label-like column found, columns = {ds.column_names}")

stsb = DatasetDict({k: _normalize_stsb_label(v) for k, v in stsb.items()})
stsb_train = stsb["train"]
stsb_validation = stsb["validation"]
stsb_test = stsb["test"]

mrpc = load_local_splits("MRPC", {
    "train": "train-mrpc.parquet",
    "validation": "validation-mrpc.parquet",
    "test": "test-mrpc.parquet",
})

qqp = load_local_splits("QQP", {
    "train": "train-qqp.parquet",
    "validation": "validation-qqp.parquet",
    "test": "test-qqp.parquet",
})

paws = load_local_splits("PAWS", {
    "train": "train-paws.parquet",
    "validation": "validation-paws.parquet",
    "test": "test-paws.parquet",
})

nli_path = os.path.join(DATA_DIR, "train-allnli.parquet")
if os.path.exists(nli_path):
    _nli_df = pd.read_parquet(nli_path)
    _renames = {}
    for want, cands in [("anchor",   ("anchor", "premise", "sent0", "query")),
                        ("positive", ("positive", "hypothesis", "entailment", "sent1", "pos")),
                        ("negative", ("negative", "contradiction", "sent2", "neg", "hard_neg"))]:
        if want not in _nli_df.columns:
            hit = next((c for c in cands if c in _nli_df.columns), None)
            if hit is None:
                raise ValueError(f"[AllNLI] no column for '{want}', found {list(_nli_df.columns)}")
            _renames[hit] = want
    if _renames:
        _nli_df = _nli_df.rename(columns=_renames)
    _nli_df = _nli_df[["anchor", "positive", "negative"]].dropna().reset_index(drop=True)
    nli_triplets_ds = HFDataset.from_pandas(_nli_df, preserve_index=False)
    print(f"[AllNLI] loaded: {len(nli_triplets_ds):,} triplets")
else:
    nli_triplets_ds = None
    print("[AllNLI] skipped -- no local file provided")

print("\n--- external datasets ready ---")
print(f"STS-B train/val/test: {len(stsb_train):,} / {len(stsb_validation):,} / {len(stsb_test):,}")
print(f"MRPC  train/val/test: {len(mrpc['train']):,} / "
      f"{len(mrpc['validation']):,} / {len(mrpc['test']):,}")
print(f"QQP   train/val/test: {len(qqp['train']):,} / "
      f"{len(qqp['validation']):,} / {len(qqp['test']):,}")
print(f"PAWS  train/val/test: {len(paws['train']):,} / "
      f"{len(paws['validation']):,} / {len(paws['test']):,}")

[STS-B] loading from local parquet...
[STS-B] loaded: train=5,749, validation=1,500, test=1,379
[MRPC] loading from local parquet...
[MRPC] loaded: train=3,668, validation=408, test=1,725
[QQP] loading from local parquet...
[QQP] loaded: train=363,846, validation=40,430, test=390,965
[PAWS] loading from local parquet...
[PAWS] loaded: train=49,401, validation=8,000, test=8,000
[AllNLI] loaded: 557,850 triplets

--- external datasets ready ---
STS-B train/val/test: 5,749 / 1,500 / 1,379
MRPC  train/val/test: 3,668 / 408 / 1,725
QQP   train/val/test: 363,846 / 40,430 / 390,965
PAWS  train/val/test: 49,401 / 8,000 / 8,000


In [9]:
def stratified_sample_indices(labels, limit, seed):
    indices = np.arange(len(labels))
    if limit is None or limit >= len(indices):
        return indices
    _, selected = train_test_split(
        indices, test_size=limit, stratify=labels, random_state=seed
    )
    return np.sort(selected)

def extract_pair_data(split, first_column, second_column, indices):
    return {
        "sentence1": [split[first_column][int(index)] for index in indices],
        "sentence2": [split[second_column][int(index)] for index in indices],
        "labels": np.asarray(
            [split["label"][int(index)] for index in indices], dtype=np.int64
        ),
    }

def make_tune_eval_split(pair_data, seed):
    indices = np.arange(len(pair_data["labels"]))
    tune_idx, eval_idx = train_test_split(
        indices, test_size=0.50, stratify=pair_data["labels"], random_state=seed
    )
    def take(idx):
        return {
            "sentence1": [pair_data["sentence1"][i] for i in idx],
            "sentence2": [pair_data["sentence2"][i] for i in idx],
            "labels": pair_data["labels"][idx],
        }
    return {"tune": take(tune_idx), "evaluation": take(eval_idx)}

mrpc_train_labels = np.asarray(mrpc["train"]["label"], dtype=np.int64)
mrpc_ckpt_idx = stratified_sample_indices(mrpc_train_labels, CFG.checkpoint_mrpc_pairs, SEED)
mrpc_checkpoint_data = make_tune_eval_split(
    extract_pair_data(mrpc["train"], "sentence1", "sentence2", mrpc_ckpt_idx), SEED
)

qqp_train_labels = np.asarray(qqp["train"]["label"], dtype=np.int64)
qqp_heldout_idx = stratified_sample_indices(qqp_train_labels, CFG.checkpoint_qqp_pairs, SEED)
qqp_checkpoint_data = make_tune_eval_split(
    extract_pair_data(qqp["train"], "question1", "question2", qqp_heldout_idx), SEED
)
qqp_excluded = set(int(i) for i in qqp_heldout_idx)                                                               
paws_train_labels = np.asarray(paws["train"]["label"], dtype=np.int64)
paws_heldout_idx = stratified_sample_indices(paws_train_labels, CFG.checkpoint_paws_pairs, SEED)
paws_checkpoint_data = make_tune_eval_split(
    extract_pair_data(paws["train"], "sentence1", "sentence2", paws_heldout_idx), SEED
)
paws_excluded = set(int(i) for i in paws_heldout_idx)

print("MRPC ckpt pairs:", len(mrpc_ckpt_idx),
      "| QQP held-out:", len(qqp_heldout_idx),
      "| PAWS held-out:", len(paws_heldout_idx))

MRPC ckpt pairs: 3000 | QQP held-out: 4000 | PAWS held-out: 3000


In [10]:
rng_data = random.Random(SEED + 17)

def build_qqp_triplets(dataset, excluded, max_pairs, rng):
    q1 = dataset["question1"]; q2 = dataset["question2"]
    labels = dataset["label"]
    hard_negative_map: Dict[str, List[str]] = {}
    positives = []
    for index in range(len(labels)):
        if index in excluded:
            continue
        if labels[index] == 0:
            hard_negative_map.setdefault(normalize_text(q1[index]), []).append(q2[index])
    for index in range(len(labels)):
        if index in excluded or labels[index] != 1:
            continue
        positives.append((q1[index], q2[index]))
    rng.shuffle(positives)
    positives = positives[:max_pairs]
    triplets = []
    for anchor, positive in positives:
        negatives = hard_negative_map.get(normalize_text(anchor))
        negative = rng.choice(negatives) if negatives else None
        triplets.append({"anchor": anchor, "positive": positive, "negative": negative})
    return triplets

def build_paws_triplets(dataset, excluded, max_pairs, rng):
    s1 = dataset["sentence1"]; s2 = dataset["sentence2"]
    labels = dataset["label"]
    groups: Dict[str, Dict[str, List[str]]] = {}
    for index in range(len(labels)):
        if index in excluded:
            continue
        key = normalize_text(s1[index])
        bucket = groups.setdefault(key, {"anchor": s1[index], "pos": [], "neg": []})
        (bucket["pos"] if labels[index] == 1 else bucket["neg"]).append(s2[index])
    triplets = []
    for bucket in groups.values():
        for positive in bucket["pos"]:
            negative = rng.choice(bucket["neg"]) if bucket["neg"] else None
            triplets.append({
                "anchor": bucket["anchor"], "positive": positive, "negative": negative
            })
    rng.shuffle(triplets)
    return triplets[:max_pairs]

def build_nli_triplets(dataset, count, rng):
    total = len(dataset)
    indices = rng.sample(range(total), min(count, total))
    subset = dataset.select(indices)
    return [
        {"anchor": a, "positive": p, "negative": n}
        for a, p, n in zip(subset["anchor"], subset["positive"], subset["negative"])
    ]

qqp_triplets, paws_triplets, stsb_pairs = [], [], []
if CFG.use_task_train_data:
    qqp_triplets = build_qqp_triplets(qqp["train"], qqp_excluded, CFG.qqp_train_pairs, rng_data)
    paws_triplets = build_paws_triplets(paws["train"], paws_excluded, CFG.paws_train_pairs, rng_data)
    stsb_pairs = [
        {"s1": a, "s2": b, "score": float(score)}
        for a, b, score in zip(
            stsb_train["sentence1"], stsb_train["sentence2"], stsb_train["label"]
        )
    ]
    
nli_triplets = build_nli_triplets(nli_triplets_ds, CFG.nli_triplet_count, rng_data)

with_hard_negative = sum(1 for t in qqp_triplets if t["negative"] is not None)
print(f"QQP triplets: {len(qqp_triplets):,} ({with_hard_negative:,} with mined hard negatives)")
print(f"PAWS triplets: {len(paws_triplets):,}")
print(f"STS-B CoSENT pairs: {len(stsb_pairs):,}")
print(f"NLI triplets: {len(nli_triplets):,}")

QQP triplets: 60,000 (6,182 with mined hard negatives)
PAWS triplets: 20,503
STS-B CoSENT pairs: 5,749
NLI triplets: 100,000


In [11]:
_POS_MAP = {"J": wordnet.ADJ, "V": wordnet.VERB, "N": wordnet.NOUN, "R": wordnet.ADV}

def _wn_pos(tag: str):
    return _POS_MAP.get(tag[0], None)

def _wordnet_lemmas(word: str, pos, antonym: bool) -> List[str]:
    results = set()
    synsets = wordnet.synsets(word, pos=pos) if pos else []
    for synset in synsets:
        for lemma in synset.lemmas():
            if antonym:
                for ant in lemma.antonyms():
                    results.add(ant.name().replace("_", " "))
            else:
                candidate = lemma.name().replace("_", " ")
                if candidate.lower() != word.lower():
                    results.add(candidate)
    return list(results)

def replace_words(sentence: str, n: int, mode: str, rng) -> Optional[str]:
    tokens = nltk.word_tokenize(sentence)
    tags = nltk.pos_tag(tokens)
    candidates = [
        (i, w, t) for i, (w, t) in enumerate(tags)
        if t.startswith("J") or t.startswith("V")
    ]
    rng.shuffle(candidates)
    replaced, count = tokens[:], 0
    for i, word, tag in candidates:
        if count >= n:
            break
        pool = _wordnet_lemmas(word, _wn_pos(tag), antonym=(mode == "antonym"))
        if pool:
            replaced[i] = rng.choice(pool)
            count += 1
    return " ".join(replaced) if count > 0 else None

def jumble_sentence(sentence: str, n_swaps: int, rng) -> Optional[str]:
    tokens = nltk.word_tokenize(sentence)
    if len(tokens) < 4:
        return None
    for _ in range(min(n_swaps, len(tokens) // 2)):
        i, j = rng.sample(range(len(tokens)), 2)
        tokens[i], tokens[j] = tokens[j], tokens[i]
    jumbled = " ".join(tokens)
    return jumbled if normalize_text(jumbled) != normalize_text(sentence) else None

# Regenerate the "jumbled" rows in train/dev/test_df using controlled 3-word
# pairwise swaps (this function), replacing the original "shuffle"-sourced
# values -- which turned out to be FULL sentence permutations (every word
# repositioned), a much easier/more obvious corruption than a 3-word swap.
# Matches jumbling's difficulty closer to antonym's (one word changed,
# everything else identical) instead of the near-trivial full-shuffle signal.
_jumble_rng = random.Random(SEED + 47)

def _regenerate_jumbled(df, rng):
    df = df.copy()
    mask = df["relation_type"] == "jumbled"
    regenerated = [jumble_sentence(a, 3, rng) for a in df.loc[mask, "anchor"]]
    original = df.loc[mask, "candidate"].tolist()
    # keep the original "shuffle" value where regeneration fails (e.g. the
    # anchor is under 4 tokens, too short for jumble_sentence to touch)
    df.loc[mask, "candidate"] = [
        new if new is not None else old for new, old in zip(regenerated, original)
    ]
    n_regenerated = sum(1 for r in regenerated if r is not None)
    n_kept_original = sum(1 for r in regenerated if r is None)
    return df, n_regenerated, n_kept_original

train_df, _tr_regen, _tr_kept = _regenerate_jumbled(train_df, _jumble_rng)
dev_df, _dev_regen, _dev_kept = _regenerate_jumbled(dev_df, _jumble_rng)
test_df, _test_regen, _test_kept = _regenerate_jumbled(test_df, _jumble_rng)

print(f"Regenerated 'jumbled' rows (max 3-word swaps, was full-permutation "
      f"'shuffle'): train={_tr_regen:,} regenerated/{_tr_kept:,} kept-original, "
      f"dev={_dev_regen:,}/{_dev_kept:,}, test={_test_regen:,}/{_test_kept:,}")

_NEGATABLE = {"is", "are", "was", "were", "do", "does", "did",
              "can", "could", "will", "would", "should", "has", "have", "had"}

def negate_sentence(sentence: str) -> Optional[str]:
    tokens = sentence.split()
    for index, token in enumerate(tokens):
        lowered = token.lower()
        if lowered in _NEGATABLE: 
            nxt = tokens[index + 1].lower() if index + 1 < len(tokens) else ""
            if nxt in {"not", "n't", "never"}:
                return None
            return " ".join(tokens[: index + 1] + ["not"] + tokens[index + 1:])
    return None

def build_augmented_relations(sentence_pool, n_anchors, seed):
    rng = random.Random(seed)
    pool = [s for s in sentence_pool if 6 <= len(s.split()) <= 28]
    rng.shuffle(pool)
    rows = []
    anchors_done = 0
    for sentence in pool:
                                                                            
                                                                           
                                
        if anchors_done >= n_anchors:
            break
        synonym_version = replace_words(sentence, 2, "synonym", rng)
        if synonym_version is None:
            continue                                                     
        rows.append({"anchor": sentence, "candidate": synonym_version,
                     "relation_type": "synonym"})
        anchors_done += 1
        antonym_version = replace_words(sentence, 2, "antonym", rng)
        if antonym_version:
            rows.append({"anchor": sentence, "candidate": antonym_version,
                         "relation_type": "antonym"})
        jumbled_version = jumble_sentence(sentence, 3, rng)
        if jumbled_version:
            rows.append({"anchor": sentence, "candidate": jumbled_version,
                         "relation_type": "jumbled"})
        negated_version = negate_sentence(sentence)
        if negated_version:
            rows.append({"anchor": sentence, "candidate": negated_version,
                         "relation_type": "negation"})
    return pd.DataFrame(rows)
print("WordNet augmentation + quad construction: CPU-heavy, prints as it goes.",
      flush=True)
if CFG.use_generic_augmentation:
    generic_pool = list(dict.fromkeys(
        [t["anchor"] for t in nli_triplets[: CFG.augmented_anchor_count * 3]]
    ))
    augmented_df = build_augmented_relations(
        generic_pool, CFG.augmented_anchor_count, SEED + 5
    )
    print(f"augmented rows: {len(augmented_df):,}, "
          f"anchors: {augmented_df['anchor'].nunique():,}")
    display(augmented_df["relation_type"].value_counts())
    train_df = pd.concat([train_df, augmented_df], ignore_index=True)
    train_df = train_df.drop_duplicates(
        subset=["anchor", "candidate", "relation_type"]
    ).reset_index(drop=True)
    print(f"train_df after augmentation: {len(train_df):,} rows, "
          f"{train_df['anchor'].nunique():,} anchors")

def build_criteria_quads(pair_list, max_quads, rng):
    pairs = list(pair_list)
    rng.shuffle(pairs)
    quads = []
    progress = tqdm(total=max_quads, desc="criteria quads", unit="quad")
    for original, paraphrase in pairs:
        if len(quads) >= max_quads:
            break
        antonym_version = replace_words(original, 2, "antonym", rng)
        jumbled_version = jumble_sentence(original, 3, rng)
        if antonym_version and jumbled_version:
            quads.append({
                "original": original,
                "paraphrase": paraphrase,
                "antonym": antonym_version,
                "jumbled": jumbled_version,
            })
            progress.update(1)
    progress.close()
    return quads

def build_relation_quads(dataframe, required, max_quads=None, seed=SEED,
                          group_col="anchor"):
    """Pivot a long-format (anchor, candidate, relation_type) dataframe --
    the custom dataset's own schema -- into per-anchor tuples containing
    exactly the relation types in `required`. Uses the first candidate found
    per (anchor, relation_type); anchors missing any required relation are
    dropped. Returns a shuffled list of (anchor_text, {relation: candidate})
    capped at max_quads."""
    grouped = {}
    for anchor, group in dataframe.groupby(group_col, sort=False):
        row = {}
        for rel in required:
            sub = group[group["relation_type"] == rel]
            if len(sub):
                row[rel] = sub["candidate"].iloc[0]
        if all(rel in row for rel in required):
            grouped[anchor] = row
    items = list(grouped.items())
    rng = random.Random(seed)
    rng.shuffle(items)
    if max_quads is not None:
        items = items[:max_quads]
    return items

_quad_rng = random.Random(SEED + 31)

# Reverted to 100% your own data. The 75/25 QQP/PAWS mix was tried and
# undone: pushing gradient share toward "criteria" via task_weights (see
# above) targets the same C3-lag problem without reintroducing PAWS/QQP's
# domain, which is what caused the earlier MR/CR downstream regression.
_own_items = build_relation_quads(
    train_df, required=("paraphrase", "antonym", "jumbled"),
    max_quads=CFG.criteria_quad_count, seed=SEED + 31,
)
criteria_quads = [
    {"original": anchor, "paraphrase": row["paraphrase"],
     "antonym": row["antonym"], "jumbled": row["jumbled"]}
    for anchor, row in _own_items
]
print(f"criteria quadruples (from your data only, train_df): "
      f"{len(criteria_quads):,}")


Regenerated 'jumbled' rows (max 3-word swaps, was full-permutation 'shuffle'): train=7,999 regenerated/0 kept-original, dev=1,000/0, test=1,001/0
WordNet augmentation + quad construction: CPU-heavy, prints as it goes.
augmented rows: 27,146, anchors: 8,000


relation_type
synonym     8000
jumbled     7998
antonym     7359
negation    3789
Name: count, dtype: int64

train_df after augmentation: 86,346 rows, 15,631 anchors
criteria quadruples (from your data only, train_df): 7,999


In [12]:
class AnchorRelationDataset(Dataset):
    def __init__(self, dataframe, hard_negative_mix, seed=SEED):
        self.seed = seed
        self.epoch = 0
        self.hard_negative_mix = hard_negative_mix
        self.anchor_map = {}
        for anchor, group in dataframe.groupby("anchor", sort=False):
            relation_map = {}
            for relation, relation_group in group.groupby("relation_type"):
                candidates = list(dict.fromkeys(
                    relation_group["candidate"].astype(str).tolist()
                ))
                if candidates:
                    relation_map[relation] = candidates
            has_positive = any(r in relation_map for r in POSITIVE_TYPES)
            has_negative = any(r in relation_map for r in HARD_NEGATIVE_TYPES)
            if has_positive and has_negative:
                self.anchor_map[anchor] = relation_map
        self.anchors = list(self.anchor_map.keys())
        if not self.anchors:
            raise ValueError("No usable training anchors.")
        print("Usable training anchors:", len(self.anchors))

    def set_epoch(self, epoch):
        self.epoch = epoch

    def __len__(self):
        return len(self.anchors)

    def __getitem__(self, index):
        rng = random.Random(self.seed + 1_000_003 * self.epoch + index)
        anchor = self.anchors[index]
        relation_map = self.anchor_map[anchor]
        relations = [r for r in RELATION_ORDER if r in relation_map]

        positive_pairs = [
            (relation, candidate)
            for relation in POSITIVE_TYPES
            if relation in relation_map
            for candidate in relation_map[relation]
        ]
        positive_relation, positive_candidate = rng.choice(positive_pairs)

        sampled = {r: rng.choice(relation_map[r]) for r in relations}
        sampled[positive_relation] = positive_candidate

        negative_options = [r for r in relations if r in HARD_NEGATIVE_TYPES]
        negative_weights = [self.hard_negative_mix.get(r, 0.1) for r in negative_options]
        negative_relation = rng.choices(negative_options, weights=negative_weights, k=1)[0]

        candidates = [sampled[r] for r in relations]
        return {
            "anchor": anchor,
            "candidates": candidates,
            "ranks": [RELATION_RANK[r] for r in relations],
            "positive_index": relations.index(positive_relation),
            "negative_index": relations.index(negative_relation),
        }

def collate_anchor_batch(batch):
    anchors, candidate_texts, owner_indices, ranks = [], [], [], []
    positive_global, negative_global = [], []
    positive_texts, negative_texts = [], []
    group_slices = []
    cursor = 0
    for owner_index, item in enumerate(batch):
        anchors.append(item["anchor"])
        candidate_count = len(item["candidates"])
        candidate_texts.extend(item["candidates"])
        owner_indices.extend([owner_index] * candidate_count)
        ranks.extend(item["ranks"])
        positive_global.append(cursor + item["positive_index"])
        negative_global.append(cursor + item["negative_index"])
        positive_texts.append(item["candidates"][item["positive_index"]])
        negative_texts.append(item["candidates"][item["negative_index"]])
        group_slices.append((cursor, cursor + candidate_count))
        cursor += candidate_count
    return {
        "anchors": anchors,
        "candidate_texts": candidate_texts,
        "owner_indices": torch.tensor(owner_indices, dtype=torch.long),
        "rank_values": torch.tensor(ranks, dtype=torch.float32),
        "positive_global_indices": torch.tensor(positive_global, dtype=torch.long),
        "negative_global_indices": torch.tensor(negative_global, dtype=torch.long),
        "positive_texts": positive_texts,
        "negative_texts": negative_texts,
        "group_slices": group_slices,
    }

class ListDataset(Dataset):
    def __init__(self, items):
        self.items = items
    def __len__(self):
        return len(self.items)
    def __getitem__(self, index):
        return self.items[index]

def collate_triplets(batch):
    return {
        "anchors": [b["anchor"] for b in batch],
        "positives": [b["positive"] for b in batch],
        "negatives": [b["negative"] for b in batch if b.get("negative")],
    }

def collate_scored_pairs(batch):
    return {
        "s1": [b["s1"] for b in batch],
        "s2": [b["s2"] for b in batch],
        "scores": [b["score"] for b in batch],
    }

In [13]:
from sentence_transformers.models import Transformer as _STTransformer
from sentence_transformers.models import Pooling as _STPooling


def build_base_model() -> SentenceTransformer:
    """MPNet's sentence embedding is standard MEAN pooling over token
    embeddings (the pooling strategy all-mpnet-base-v2 was trained with) --
    unlike SimCSE, no custom pooler-head module is needed here. Plain
    sentence_transformers Transformer + Pooling modules are correct and
    sufficient."""
    print(f"building {BASE_MODEL} (first call downloads the checkpoint; "
          "later calls hit the cache)...", flush=True)
    transformer = _STTransformer(BASE_MODEL, max_seq_length=CFG.max_seq_length)
    pooling = _STPooling(
        transformer.get_word_embedding_dimension(), pooling_mode="mean"
    )
    model = SentenceTransformer(modules=[transformer, pooling], device=str(DEVICE))
    model.max_seq_length = CFG.max_seq_length
    print("model ready (mean pooling).", flush=True)
    return model

def set_auto_model(st_model, new_module) -> None:
    transformer_module = st_model[0]
    if "auto_model" in transformer_module._modules:
        transformer_module._modules["auto_model"] = new_module
        return
    for slot, module in transformer_module._modules.items():
        if hasattr(module, "config") and hasattr(module, "forward"):
            transformer_module._modules[slot] = new_module
            return
    raise RuntimeError(
        "Cannot locate the auto_model submodule slot. _modules keys: "
        f"{list(transformer_module._modules.keys())}"
    )

def build_lora_model() -> SentenceTransformer:
    model = build_base_model()
    lora_config = LoraConfig(
        task_type=TaskType.FEATURE_EXTRACTION,
        inference_mode=False,
        r=CFG.lora_r,
        lora_alpha=CFG.lora_alpha,
        lora_dropout=CFG.lora_dropout,
        bias="none",
        target_modules=list(CFG.lora_targets),
    )

    peft_auto_model = PeftModelForFeatureExtraction(model[0].auto_model, lora_config)
    set_auto_model(model, peft_auto_model)                                      

    kind = type(model[0].auto_model).__name__
    print("Wrapped model type:", kind)
    if kind != "PeftModelForFeatureExtraction":
        raise RuntimeError(
            f"PEFT wrapper did not stick: model[0].auto_model is still {kind}. "
            "The adapter would be inert. Do not train."
        )

    peft_auto_model.print_trainable_parameters()
    lora_names = [
        name for name, _ in model[0].auto_model.named_parameters()
        if "lora_" in name.lower()
    ]
    if not lora_names:
        raise RuntimeError("LoRA was not attached to the base model.")
    print(f"LoRA parameter tensors found: {len(lora_names)}")
    trainable_lora = [
        name for name, param in model.named_parameters()
        if param.requires_grad and "lora_" in name.lower()
    ]
    if not trainable_lora:
        raise RuntimeError(
            "LoRA params exist on the PEFT object but are NOT visible through "
            "model.named_parameters(). The optimizer would train nothing."
        )
    print(f"LoRA tensors visible to the optimizer: {len(trainable_lora)}")
    try:
        peft_auto_model.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={"use_reentrant": False}
        )
        if hasattr(peft_auto_model, "enable_input_require_grads"):
            peft_auto_model.enable_input_require_grads()
        print("Gradient checkpointing enabled.")
    except (TypeError, ValueError) as error:
        print(f"Gradient checkpointing unavailable ({error}); continuing without it.")
    return model

def assert_peft_intact(st_model, where=""):
    kind = type(st_model[0].auto_model).__name__
    if kind != "PeftModelForFeatureExtraction":
        raise RuntimeError(
            f"PEFT wrapper missing at {where!r}: model[0].auto_model is {kind}. "
        )
    return True

def scaled_lora_state(state, lam):
    return {
        key: (value * lam if "lora_B" in key else value.clone())
        for key, value in state.items()
    }

def load_lora_into_fresh_model(state, lam=1.0):
    fresh = build_lora_model()
    assert_peft_intact(fresh, "load_lora_into_fresh_model")
    set_peft_model_state_dict(fresh[0].auto_model, scaled_lora_state(state, lam))
    fresh.eval()
    return fresh

def amp_context():
    if DEVICE.type != "cuda":
        return nullcontext()
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    return torch.autocast(device_type="cuda", dtype=dtype)

def forward_embeddings(model, texts, require_grad):
    if not texts:
        raise ValueError("Cannot encode an empty text list.")
    features = batch_to_device(model.tokenize(texts), DEVICE)
    gradient_context = torch.enable_grad() if require_grad else torch.no_grad()
    with gradient_context:
        with amp_context():
            embeddings = model(features)["sentence_embedding"]
    return F.normalize(embeddings.float(), p=2, dim=-1)

@torch.no_grad()
def encode_numpy(model, texts, batch_size=None):
    model.eval()
    return model.encode(
        texts,
        batch_size=batch_size or CFG.encode_batch_size,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype(np.float32)

def get_linear_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps):
    def learning_rate_lambda(step):
        if step < num_warmup_steps:
            return step / max(1, num_warmup_steps)
        return max(
            0.0,
            (num_training_steps - step) / max(1, num_training_steps - num_warmup_steps),
        )
    return LambdaLR(optimizer, learning_rate_lambda)

In [14]:
print("[pre-flight 1/3] building the LoRA model...", flush=True)
_pf_lora = build_lora_model()
_kind = type(_pf_lora[0].auto_model).__name__
print("lora model[0].auto_model:", _kind)
if _kind != "PeftModelForFeatureExtraction":
    raise RuntimeError(f"PEFT wrapper not attached: auto_model is {_kind}")

print("[pre-flight 2/3] encoding the probe (lora_B=0, so this IS the base "
      "model's output)...", flush=True)
_probe = ["a quick sanity probe sentence", "an unrelated second sentence"]
_v_before = _pf_lora.encode(_probe, normalize_embeddings=True)

print("[pre-flight 3/3] perturbing lora_B and re-encoding...", flush=True)
with torch.no_grad():
    _touched = 0
    for _name, _param in _pf_lora.named_parameters():
        if "lora_B" in _name:
            _param.add_(torch.randn_like(_param) * 0.05)
            _touched += 1
print(f"perturbed {_touched} lora_B tensors")

_v_after = _pf_lora.encode(_probe, normalize_embeddings=True)
_d_pert = float(np.abs(_v_before - _v_after).mean())
print(f"embedding delta after perturbing lora_B: {_d_pert:.8f}")

if _touched == 0:
    raise RuntimeError("No lora_B tensors found in model.named_parameters().")
if _d_pert < 1e-6:
    raise RuntimeError(
        "PERTURBING lora_B DID NOT CHANGE THE EMBEDDINGS. The LoRA adapter is "
        "NOT in the forward path. Training would be a no-op. Fix set_auto_model()."
    )
print("\nPRE-FLIGHT PASSED: the LoRA adapter is in the forward path.")

del _pf_lora
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

[pre-flight 1/3] building the LoRA model...
building sentence-transformers/all-mpnet-base-v2 (first call downloads the checkpoint; later calls hit the cache)...


model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

model ready (mean pooling).
Wrapped model type: PeftModelForFeatureExtraction
trainable params: 884,736 || all params: 110,371,200 || trainable%: 0.8016
LoRA parameter tensors found: 48
LoRA tensors visible to the optimizer: 48
Gradient checkpointing unavailable (MPNetModel does not support gradient checkpointing.); continuing without it.
lora model[0].auto_model: PeftModelForFeatureExtraction
[pre-flight 2/3] encoding the probe (lora_B=0, so this IS the base model's output)...
[pre-flight 3/3] perturbing lora_B and re-encoding...
perturbed 24 lora_B tensors
embedding delta after perturbing lora_B: 0.01070107

PRE-FLIGHT PASSED: the LoRA adapter is in the forward path.


In [15]:
def masked_infonce(anchor_embeddings, candidate_embeddings, candidate_texts,
                   anchors, positive_texts, temperature):
    logits = (anchor_embeddings @ candidate_embeddings.T) / temperature
    batch_size = anchor_embeddings.size(0)
    for anchor_index in range(batch_size):
        protected = {
            normalize_text(anchors[anchor_index]),
            normalize_text(positive_texts[anchor_index]),
        }
        for candidate_index, candidate_text in enumerate(candidate_texts):
            if candidate_index == anchor_index:
                continue
            if normalize_text(candidate_text) in protected:
                logits[anchor_index, candidate_index] = torch.finfo(logits.dtype).min
    labels = torch.arange(batch_size, device=anchor_embeddings.device)
    return F.cross_entropy(logits, labels)

def relation_step(model, batch):
    """Relation task loss -- masked InfoNCE only.

    within_anchor_rank_loss() (the pairwise ranking loss enforcing
    RELATION_RANK ordering within each anchor group -- paraphrase >
    antonym > jumbled > distinct, etc.) has been REMOVED. This step now
    trains purely on contrastive separation (masked_infonce: pull the
    true positive together, push everything else apart within the
    batch), with no explicit within-anchor ordering signal.

    batch["owner_indices"]/["rank_values"]/["group_slices"] are still
    built by collate_anchor_batch() (unused here now, but harmless to
    leave -- see the collate function's own docstring) and RELATION_RANK
    is still used elsewhere (evaluate_relation_alignment's reporting,
    unrelated to this loss)."""
    positive_indices = batch["positive_global_indices"].to(DEVICE)
    negative_indices = batch["negative_global_indices"].to(DEVICE)

    anchor_embeddings = forward_embeddings(model, batch["anchors"], True)
    candidate_embeddings = forward_embeddings(model, batch["candidate_texts"], True)
    positive_embeddings = candidate_embeddings[positive_indices]
    negative_embeddings = candidate_embeddings[negative_indices]

    contrast_candidates = torch.cat([positive_embeddings, negative_embeddings], dim=0)
    contrast_texts = batch["positive_texts"] + batch["negative_texts"]
    loss_nce = masked_infonce(
        anchor_embeddings, contrast_candidates, contrast_texts,
        batch["anchors"], batch["positive_texts"], CFG.temperature,
    )
    return loss_nce

def triplet_step(model, batch):
    """Plain InfoNCE triplet step, used for QQP / PAWS triplets."""
    anchor_embeddings = forward_embeddings(model, batch["anchors"], True)
    positive_embeddings = forward_embeddings(model, batch["positives"], True)
    candidate_texts = list(batch["positives"])
    candidate_embeddings = positive_embeddings
    if batch["negatives"]:
        negative_embeddings = forward_embeddings(model, batch["negatives"], True)
        candidate_embeddings = torch.cat([positive_embeddings, negative_embeddings], dim=0)
        candidate_texts = candidate_texts + list(batch["negatives"])
    return masked_infonce(
        anchor_embeddings, candidate_embeddings, candidate_texts,
        batch["anchors"], batch["positives"], CFG.temperature,
    )

def nli_step(model, batch):
    """NLI triplet step (anchor=premise, positive=entailment, negative=contradiction).

    L_NLI = L_InfoNCE + CFG.nli_sep_weight * L_sep
    gap = sim(anchor, entailment) - sim(anchor, contradiction)
    L_sep = relu(CFG.nli_margin - gap).mean()

    This is a HINGE, not a raw linear separation term: once the gap clears
    CFG.nli_margin, the loss for that example is exactly 0 and gradient stops
    pushing it any further. A raw `-(gap)` term (what this used to be) has no
    such stopping point -- it keeps enlarging the gap indefinitely even after
    "enough" separation is reached, which can over-shape the embedding
    geometry in a way that fights the smooth, graded similarity ordering
    STS-B needs. The hinge only pushes while the constraint is actually
    violated, same as criteria_step()'s C3/C4 margins.
    """
    anchor_embeddings = forward_embeddings(model, batch["anchors"], True)
    positive_embeddings = forward_embeddings(model, batch["positives"], True)                 
    candidate_texts = list(batch["positives"])
    candidate_embeddings = positive_embeddings

    loss_sep = anchor_embeddings.new_tensor(0.0)
    if batch["negatives"]:
        negative_embeddings = forward_embeddings(model, batch["negatives"], True)                 
        candidate_embeddings = torch.cat([positive_embeddings, negative_embeddings], dim=0)
        candidate_texts = candidate_texts + list(batch["negatives"])

        sim_entailment = (anchor_embeddings * positive_embeddings).sum(dim=-1)
        sim_contradiction = (anchor_embeddings * negative_embeddings).sum(dim=-1)
        gap = sim_entailment - sim_contradiction
        loss_sep = F.relu(CFG.nli_margin - gap).mean()

    loss_nce = masked_infonce(
        anchor_embeddings, candidate_embeddings, candidate_texts,
        batch["anchors"], batch["positives"], CFG.temperature,
    )
    return loss_nce + CFG.nli_sep_weight * loss_sep

def collate_quads(batch):
    return {
        "original": [b["original"] for b in batch],
        "paraphrase": [b["paraphrase"] for b in batch],
        "antonym": [b["antonym"] for b in batch],
        "jumbled": [b["jumbled"] for b in batch],
    }

def sample_criteria_epsilon() -> float:
    """Minimum margin for the C3/C4 hinges, sampled fresh each step:

        eps ~ Uniform(CFG.criteria_eps_min, CFG.criteria_eps_max)  ->  [0.05, 0.10]

    Both ends are strictly positive. At eps < 0 the hinge returns zero loss
    even when the antonym/jumbled sentence is rated MORE similar than the
    paraphrase -- i.e. it would reward exactly the failure the criterion
    exists to catch. The upper end matches rank_margin, so no single step
    demands a larger gap than the ranking loss itself.
    """
    assert 0.0 <= CFG.criteria_eps_min <= CFG.criteria_eps_max, (
        f"Invalid eps range: [{CFG.criteria_eps_min}, {CFG.criteria_eps_max}]"
    )
    return random.uniform(CFG.criteria_eps_min, CFG.criteria_eps_max)

def criteria_step(model, batch):
    """Dedicated C3/C4 margin loss: enforces that paraphrases sit closer to
    the anchor than antonym-replaced (C3) or jumbled (C4) versions, with a
    real stopping margin -- unlike a raw linear separation term, this hinge
    stops pushing once the eps margin is cleared, so it does not keep
    over-shaping the embedding geometry once the constraint is satisfied.

    Just the two core relative margins -- no per-class similarity ceilings,
    no absolute positive floor, no extra antonym-vs-jumbled ordering term.
    Those were auxiliary regularizers on top of the actual C3/C4 criteria
    (which are purely relative: paraphrase vs. antonym, paraphrase vs.
    jumbled), not requirements of the criteria themselves, so they're gone.

    Taxonomy is C1=semantic distinction, C2=synonym replacement,
    C3=antonym replacement, C4=sentence jumbling. (The old "C3/C5" labeling
    is retired: C5=jumbling was folded into C4, and the old C4=negation
    criterion was dropped.)
    """
    embeddings_original = forward_embeddings(model, batch["original"], True)
    embeddings_paraphrase = forward_embeddings(model, batch["paraphrase"], True)
    embeddings_antonym = forward_embeddings(model, batch["antonym"], True)
    embeddings_jumbled = forward_embeddings(model, batch["jumbled"], True)

    sim_para = (embeddings_original * embeddings_paraphrase).sum(dim=-1)
    sim_ant = (embeddings_original * embeddings_antonym).sum(dim=-1)
    sim_jum = (embeddings_original * embeddings_jumbled).sum(dim=-1)

                                                                
                                                       
    eps = sample_criteria_epsilon()
    loss = (
        CFG.criteria_antonym_weight * F.relu(eps - (sim_para - sim_ant)).mean()
        + CFG.criteria_jumble_weight * F.relu(eps - (sim_para - sim_jum)).mean()
    )
    return loss

def cosent_step(model, batch):
    embeddings1 = forward_embeddings(model, batch["s1"], True)
    embeddings2 = forward_embeddings(model, batch["s2"], True)
    cosine = (embeddings1 * embeddings2).sum(dim=-1)
    scores = torch.tensor(batch["scores"], dtype=torch.float32, device=cosine.device)
    difference = CFG.cosent_scale * (cosine[:, None] - cosine[None, :])
    mask = scores[:, None] < scores[None, :]
    if not mask.any():
        return cosine.new_tensor(0.0)
    violations = difference[mask]
    return torch.logsumexp(
        torch.cat([violations.new_zeros(1), violations], dim=0), dim=0
    )


In [16]:
def score_dataframe_pairs(model, dataframe):
    scored = dataframe[["anchor", "candidate", "relation_type"]].copy()
    unique_texts = list(dict.fromkeys(
        scored["anchor"].tolist() + scored["candidate"].tolist()
    ))
    embeddings = encode_numpy(model, unique_texts)
    embedding_map = {text: embeddings[index] for index, text in enumerate(unique_texts)}
    scored["cosine"] = [
        float(np.dot(embedding_map[row.anchor], embedding_map[row.candidate]))
        for row in scored.itertuples(index=False)
    ]
    return scored

In [17]:
def evaluate_relation_alignment(model, dataframe):
    scored = score_dataframe_pairs(model, dataframe)
    table = scored.groupby(["anchor", "relation_type"], as_index=False)["cosine"].mean()
    by_anchor = {
        anchor: dict(zip(group["relation_type"], group["cosine"]))
        for anchor, group in table.groupby("anchor")
    }
    reported = TARGET_EVAL_RELATIONS + EXTRA_REPORT_RELATIONS
    per_relation_correct = {r: [] for r in reported}
    all_outcomes = []
    for relation_scores in by_anchor.values():
        relations = list(relation_scores.keys())
        for first_index, first_relation in enumerate(relations):
            for second_relation in relations[first_index + 1:]:
                first_rank = RELATION_RANK[first_relation]
                second_rank = RELATION_RANK[second_relation]
                if first_rank == second_rank:
                    continue
                first_score = relation_scores[first_relation]
                second_score = relation_scores[second_relation]
                correct = (
                    float(first_score > second_score)
                    if first_rank > second_rank
                    else float(first_score < second_score)
                )
                all_outcomes.append(correct)
                if first_relation in per_relation_correct:
                    per_relation_correct[first_relation].append(correct)
                if second_relation in per_relation_correct:
                    per_relation_correct[second_relation].append(correct)
    per_relation = {
        relation: float(np.mean(values)) if values else float("nan")
        for relation, values in per_relation_correct.items()
    }

    target_values = [
        per_relation[r] for r in TARGET_EVAL_RELATIONS
        if not np.isnan(per_relation.get(r, float("nan")))
    ]
    return {
        "macro_target_alignment": (
            float(np.mean(target_values)) if target_values else float("nan")
        ),
        "overall_pairwise_alignment": float(np.mean(all_outcomes)) if all_outcomes else float("nan"),
        "per_relation_alignment": per_relation,
        "mean_cosine_by_relation": scored.groupby("relation_type")["cosine"].mean().to_dict(),
    }

In [18]:
def print_alignment(title, metrics):
    print(f"\n{title}")
    print("macro alignment:", round(metrics["macro_target_alignment"], 4))
    print("overall pairwise:", round(metrics["overall_pairwise_alignment"], 4))
    print("  -- criteria relations (in macro) --")
    for relation in TARGET_EVAL_RELATIONS:
        value = metrics["per_relation_alignment"].get(relation)
        print(
            f"  {relation:14s}: {value:.4f}"
            if value is not None and not np.isnan(value)
            else f"  {relation:14s}: n/a"
        )
    print("  -- other relations (reported only) --")
    for relation in EXTRA_REPORT_RELATIONS:
        value = metrics["per_relation_alignment"].get(relation)
        if value is not None and not np.isnan(value):
            print(f"  {relation:14s}: {value:.4f}")

def evaluate_stsb(model, split, limit=None):
    sentence1 = list(split["sentence1"])
    sentence2 = list(split["sentence2"])
    labels = np.asarray(split["label"], dtype=np.float32)
    if limit is not None:
        sentence1, sentence2, labels = sentence1[:limit], sentence2[:limit], labels[:limit]
    embeddings1 = encode_numpy(model, sentence1)
    embeddings2 = encode_numpy(model, sentence2)
    cosine = np.sum(embeddings1 * embeddings2, axis=1)
    correlation = spearmanr(cosine, labels).correlation
    if np.isnan(correlation):
        raise RuntimeError("STS-B Spearman correlation is NaN.")
    return float(correlation)

def cosine_for_pair_data(model, pair_data):
    embeddings1 = encode_numpy(model, pair_data["sentence1"])
    embeddings2 = encode_numpy(model, pair_data["sentence2"])
    return np.sum(embeddings1 * embeddings2, axis=1)

def tune_cosine_threshold(cosine_scores, labels, metric):
    best_threshold = float(cosine_scores.min())
    best_metric = -1.0
    for threshold in np.unique(cosine_scores):
        predictions = (cosine_scores > threshold).astype(np.int64)
        current_metric = (
            f1_score(labels, predictions, zero_division=0)
            if metric == "f1"
            else accuracy_score(labels, predictions)
        )
        if current_metric > best_metric:
            best_metric = current_metric
            best_threshold = float(threshold)
    return best_threshold

def evaluate_prepared_pair_data(model, prepared_data):
    tune_cosine = cosine_for_pair_data(model, prepared_data["tune"])
    tune_labels = prepared_data["tune"]["labels"]
    f1_threshold = tune_cosine_threshold(tune_cosine, tune_labels, metric="f1")
    accuracy_threshold = tune_cosine_threshold(tune_cosine, tune_labels, metric="accuracy")
    evaluation_cosine = cosine_for_pair_data(model, prepared_data["evaluation"])
    evaluation_labels = prepared_data["evaluation"]["labels"]
    f1_predictions = (evaluation_cosine > f1_threshold).astype(np.int64)
    accuracy_predictions = (evaluation_cosine > accuracy_threshold).astype(np.int64)
    return {
        "f1_threshold": f1_threshold,
        "accuracy_threshold": accuracy_threshold,
        "f1": f1_score(evaluation_labels, f1_predictions, zero_division=0),
        "acc": accuracy_score(evaluation_labels, accuracy_predictions),
    }

In [19]:
def _build_criteria_probe(n_pairs, seed=SEED):
    """Source the C3/C4 probe's quadruples from the custom dataset's dev
    split (dev_df: anchor, candidate, relation_type) instead of
    mrpc/qqp/paws validation. dev_df is verified anchor-disjoint from
    train_df (see Cell 6's disjointness check), so this probe stays
    disjoint from the data criteria_quads was built from, without needing
    any WordNet-generated antonym/jumbled -- dev_df already has real ones.
    """
    items = build_relation_quads(
        dev_df, required=("paraphrase", "antonym", "jumbled"),
        max_quads=n_pairs, seed=seed + 999,
    )
    return [
        {"original": anchor, "paraphrase": row["paraphrase"],
         "antonym": row["antonym"], "jumbled": row["jumbled"]}
        for anchor, row in items
    ]

CRITERIA_PROBE = _build_criteria_probe(CFG.criteria_probe_pairs)
print(f"criteria probe quadruples (from custom dataset, dev_df): "
      f"{len(CRITERIA_PROBE)}")

criteria probe quadruples (from custom dataset, dev_df): 1000


In [20]:
def evaluate_criteria_margins(model, probe=None, eps=None):
    """Return the C3 and C4 margins and pass-rates AT eps -- the actual paper
    conditions, measured mid-training. (C4 here is the sentence-jumbling
    criterion -- formerly labeled C5 before the taxonomy was collapsed to
    C1-C4.)"""
    probe = probe or CRITERIA_PROBE
    eps = CFG.criteria_eps if eps is None else eps
    originals = [q["original"] for q in probe]
    paraphrases = [q["paraphrase"] for q in probe]
    antonyms = [q["antonym"] for q in probe]
    jumbles = [q["jumbled"] for q in probe]

    e_orig = encode_numpy(model, originals)
    e_para = encode_numpy(model, paraphrases)
    e_ant = encode_numpy(model, antonyms)
    e_jum = encode_numpy(model, jumbles)

    sim_para = np.sum(e_orig * e_para, axis=1)
    sim_ant = np.sum(e_orig * e_ant, axis=1)
    sim_jum = np.sum(e_orig * e_jum, axis=1)

    m_c3 = sim_para - sim_ant
    m_c4 = sim_para - sim_jum
    return {
        "c3_margin": float(np.mean(m_c3)),
        "c4_margin": float(np.mean(m_c4)),
        "c3_pass": float(np.mean(m_c3 > eps)),
        "c4_pass": float(np.mean(m_c4 > eps)),
        "avg_para_sim": float(np.mean(sim_para)),
        "avg_ant_sim": float(np.mean(sim_ant)),
        "avg_jum_sim": float(np.mean(sim_jum)),
    }

def evaluate_embedding_geometry(model, dataframe=None):
    """Absolute-similarity diagnostic: are similar sentences actually CLOSE and
    opposites actually FAR, in absolute cosine terms?

    The ranking/alignment metrics only check ORDERING, so a model whose whole
    similarity range has collapsed can still score well on them. This reports
    the mean absolute similarity per rank class.

    Watch `equiv_minus_opposite` -- it is the binding constraint (are similar
    sentences really closer than opposites?) and the one that failed on SICK.
    `equiv_minus_distinct` is the easy boundary, reported only as a range check.
    """
    if dataframe is None:
        dataframe = dev_df
    scored = score_dataframe_pairs(model, dataframe)
    scored["rank"] = scored["relation_type"].map(RELATION_RANK)
    scored = scored.dropna(subset=["rank"])
    by_rank = scored.groupby("rank")["cosine"].mean()                                                                                       
    result = {
        "equiv_sim": float(by_rank.get(3, float("nan"))),                                    
        "opposite_sim": float(by_rank.get(2, float("nan"))),                                  
        "jumbled_sim": float(by_rank.get(1, float("nan"))),            
        "distinct_sim": float(by_rank.get(0, float("nan"))),                           
    }                                                                   
    result["equiv_minus_opposite"] = result["equiv_sim"] - result["opposite_sim"]
    result["opposite_minus_jumbled"] = result["opposite_sim"] - result["jumbled_sim"]
    result["equiv_minus_distinct"] = result["equiv_sim"] - result["distinct_sim"]
                                                                             
                                                
    result["equiv_beats_opposite"] = bool(
        result["equiv_minus_opposite"] > CFG.rank_margin
    )
                                                                    
    ordered = [result["equiv_sim"], result["opposite_sim"],
               result["jumbled_sim"], result["distinct_sim"]]
    finite = [v for v in ordered if v == v]
    result["ordering_ok"] = all(a >= b for a, b in zip(finite, finite[1:]))
    return result

def run_checkpoint_eval(model, step=None):
    model.eval()
    alignment = evaluate_relation_alignment(model, dev_df)
    # 500 shuffled from the ~1000-anchor dev pool each eval, seeded by step
    # -- not the same fixed set every time. Falls back to the static
    # CRITERIA_PROBE when step is None (the one-time baseline call).
    probe = (_build_criteria_probe(CFG.criteria_probe_eval_pairs, seed=SEED + 999 + step)
             if step is not None else CRITERIA_PROBE)
    criteria = evaluate_criteria_margins(model, probe=probe)
    geometry = evaluate_embedding_geometry(model)
    return {
        "geometry": geometry,                                                             
        "polarity": geometry["equiv_minus_distinct"],
        "alignment": alignment["macro_target_alignment"],
        "alignment_full": alignment,
        "criteria": criteria,
        "c3_pass": criteria["c3_pass"],
        "c4_pass": criteria["c4_pass"],
        "criteria_pass": 0.5 * (criteria["c3_pass"] + criteria["c4_pass"]),
        "stsb": evaluate_stsb(model, stsb_validation),
        "mrpc_f1": evaluate_prepared_pair_data(model, mrpc_checkpoint_data)["f1"],
        "qqp_f1": evaluate_prepared_pair_data(model, qqp_checkpoint_data)["f1"],
        "paws_acc": evaluate_prepared_pair_data(model, paws_checkpoint_data)["acc"],
    }

def selection_key(metrics, baseline):
    """C3/C4 are a HARD CONSTRAINT; among checkpoints that satisfy them,
    selection maximizes a BALANCED downstream score -- not the raw mean.

    Why not the raw mean of deltas: the four downstream metrics have wildly
    different headroom over the baseline. PAWS starts low (the pretrained base
    is typically weak on adversarial paraphrase) so its deltas can run large,
    while STS-B starts at ~0.88 and moves +-0.01. Averaging raw deltas made
    PAWS ~83% of the score, so "downstream improved" and "polarity kept
    increasing" became the same signal: the run kept being rewarded for
    stretching the similarity range, STS-B quietly decayed, and early
    stopping never fired because the score never stopped rising.

    balanced_delta fixes this by (a) normalizing each delta by that metric's
    headroom (1 - baseline) so all four sit on a comparable scale, then
    (b) taking the WORST metric rather than the mean. No single metric can
    dominate, and any metric going backwards immediately caps the score.
    That makes it a polarity brake as well: past the balanced point, extra
    polarity shows up as STS-B decay, which now drags the score down instead
    of being masked by PAWS gains -- so patience can actually trigger.

    mean_delta is still returned for logging/back-compat, but no longer
    drives selection.
    """
    raw = {
        "stsb": metrics["stsb"] - baseline["stsb"],
        "mrpc_f1": metrics["mrpc_f1"] - baseline["mrpc_f1"],
        "qqp_f1": metrics["qqp_f1"] - baseline["qqp_f1"],
        "paws_acc": metrics["paws_acc"] - baseline["paws_acc"],
    }
    deltas = [raw["stsb"], raw["mrpc_f1"], raw["qqp_f1"], raw["paws_acc"]]
    mean_delta = float(np.mean(deltas))                                                              
    normalized = {
        name: value / max(1e-6, 1.0 - baseline[name]) for name, value in raw.items()
    }
    balanced_delta = float(min(normalized.values()))

    criteria_pass = metrics["criteria_pass"]
    criteria_ok = metrics["c3_pass"] > 0.5 and metrics["c4_pass"] > 0.5
    no_regression = all(d >= -CFG.delta_tolerance for d in deltas)                                                             
    polarity_ok = metrics["polarity"] <= CFG.polarity_ceiling

    key = (
        int(criteria_ok and no_regression and polarity_ok),
        int(criteria_ok and polarity_ok),                                          
        int(criteria_ok),                                                                                                                        
        round(criteria_pass, 4) if not criteria_ok else 0.0,
        balanced_delta,
    )
    return key, mean_delta, deltas, balanced_delta, normalized


In [21]:
print("Computing baseline reference metrics...")
baseline_probe = build_base_model()
BASELINE = run_checkpoint_eval(baseline_probe)
print(f"Baseline dev alignment: {BASELINE['alignment']:.4f}")
print(f"Baseline STS-B:         {BASELINE['stsb']:.4f}")
print(f"Baseline MRPC-F1:       {BASELINE['mrpc_f1']:.4f}")
print(f"Baseline QQP-F1:        {BASELINE['qqp_f1']:.4f}")
print(f"Baseline PAWS-acc:      {BASELINE['paws_acc']:.4f}")

del baseline_probe
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Computing baseline reference metrics...
building sentence-transformers/all-mpnet-base-v2 (first call downloads the checkpoint; later calls hit the cache)...
model ready (mean pooling).
Baseline dev alignment: 0.7077
Baseline STS-B:         0.8810
Baseline MRPC-F1:       0.8345
Baseline QQP-F1:        0.7587
Baseline PAWS-acc:      0.6273


In [22]:
def infinite_batches(loader):
    """Cycle a loader forever, advancing the dataset's epoch counter on each
    pass. AnchorRelationDataset.__getitem__ seeds its RNG with self.epoch, so
    without this the SAME candidate/negative sampling was replayed on every
    pass -- set_epoch() existed but was never called, silently disabling the
    per-epoch resampling it was written to provide."""
    epoch = 0
    while True:
        dataset = getattr(loader, "dataset", None)
        if hasattr(dataset, "set_epoch"):
            dataset.set_epoch(epoch)
        for batch in loader:
            yield batch
        epoch += 1                                                                            
RESUME_CHECKPOINT_PATH = Path(CFG.output_dir) / CFG.resume_checkpoint_filename

def checkpoint_fingerprint():
    """Config values that change what training actually optimizes. A resume
    checkpoint written under different values is not safe to continue from."""
    return {
        "base_model": BASE_MODEL,
        "max_steps": CFG.max_steps,
        "lora_r": CFG.lora_r,
        "lora_alpha": CFG.lora_alpha,
        "rank_margin": CFG.rank_margin,
        "nli_sep_weight": CFG.nli_sep_weight,
        "nli_margin": CFG.nli_margin,
        "polarity_ceiling": CFG.polarity_ceiling,
        "criteria_antonym_weight": CFG.criteria_antonym_weight,
        "criteria_jumble_weight": CFG.criteria_jumble_weight,
        "task_weights": tuple(sorted(CFG.task_weights.items())),                               
        "criteria_eps": CFG.criteria_eps,
        "selection_scheme": "balanced_delta_v1",
        # Bumped: relation_step()'s objective changed (ranking loss
        # removed, masked_infonce only) without any CFG field value
        # itself changing, so this is the only thing that makes an old
        # resume checkpoint (trained WITH the ranking loss) get correctly
        # flagged as incompatible instead of silently resuming under a
        # different objective than the one it was actually trained with.
        "relation_step_version": "infonce_only_no_rank_loss_v1",
    }

def save_resume_checkpoint(path, step, model, optimizer, scheduler, scaler,
                            task_rng, running, counts, history,
                            best_key, best_state, best_step, evals_since_improvement=0):
    """Atomically write a full training-resume checkpoint to /kaggle/working."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = {
        "step": step,
        "config_fingerprint": checkpoint_fingerprint(),
        "lora_state": {
            k: v.detach().cpu().clone()
            for k, v in get_peft_model_state_dict(model[0].auto_model).items()
        },
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "scaler_state": scaler.state_dict() if scaler.is_enabled() else None,
        "task_rng_state": task_rng.getstate(),
        "python_random_state": random.getstate(),
        "numpy_random_state": np.random.get_state(),
        "torch_random_state": torch.get_rng_state(),
        "torch_cuda_random_state": (
            torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
        ),
        "running": running,
        "counts": counts,
        "history": history,
        "best_key": best_key,
        "best_state": best_state,
        "best_step": best_step,
        "evals_since_improvement": evals_since_improvement,
    }
                                                                       
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, tmp_path)
    tmp_path.replace(path)
    print(f"[checkpoint] step {step}: resume checkpoint saved -> {path}")

def load_resume_checkpoint(path, model, optimizer, scheduler, scaler):
    """Restore model/optimizer/scheduler/RNG state from a resume checkpoint.
    Returns (payload_dict, task_rng)."""
    path = Path(path)
    payload = torch.load(path, map_location=DEVICE, weights_only=False)                                                                    
    saved_fp = payload.get("config_fingerprint")
    current_fp = checkpoint_fingerprint()
    if (saved_fp is not None and saved_fp != current_fp
            and not CFG.allow_incompatible_resume):
        mismatched = [k for k in current_fp if current_fp.get(k) != saved_fp.get(k)]
        raise RuntimeError(
            "Resume checkpoint was written under a DIFFERENT configuration.\n"
            f"  changed: {mismatched}\n"
            f"  saved:   { {k: saved_fp.get(k) for k in mismatched} }\n"
            f"  current: { {k: current_fp.get(k) for k in mismatched} }\n"
            "Delete the checkpoint to train fresh, or set "
            "CFG.allow_incompatible_resume = True to override."
        )

    set_peft_model_state_dict(
        model[0].auto_model,
        {k: v.to(DEVICE) for k, v in payload["lora_state"].items()},
    )
    optimizer.load_state_dict(payload["optimizer_state"])
    scheduler.load_state_dict(payload["scheduler_state"])
    if scaler.is_enabled() and payload.get("scaler_state") is not None:
        scaler.load_state_dict(payload["scaler_state"])

    task_rng = random.Random()
    task_rng.setstate(payload["task_rng_state"])
    random.setstate(payload["python_random_state"])
    np.random.set_state(payload["numpy_random_state"])                                                                        
    try:
        torch_state = payload["torch_random_state"]
        if torch.is_tensor(torch_state):
            torch_state = torch_state.detach().cpu().to(torch.uint8)
        torch.set_rng_state(torch_state)
    except Exception as exc:
        print(f"[checkpoint] could not restore torch RNG state ({exc}); "
              "continuing with a fresh torch RNG.")
    if torch.cuda.is_available() and payload.get("torch_cuda_random_state") is not None:
        try:
            torch.cuda.set_rng_state_all([
                s.detach().cpu().to(torch.uint8) if torch.is_tensor(s) else s
                for s in payload["torch_cuda_random_state"]
            ])
        except Exception as exc:
            print(f"[checkpoint] could not restore CUDA RNG state ({exc}); "
                  "continuing with a fresh CUDA RNG.")

    print(f"[checkpoint] resumed from step {payload['step']} ({path})")
    return payload, task_rng

def train_multitask():
    seed_everything(SEED)
    model = build_lora_model()

    relation_dataset = AnchorRelationDataset(train_df, HARD_NEGATIVE_MIX)
    loaders = {
        "relation": DataLoader(
            relation_dataset, batch_size=CFG.relation_batch_size, shuffle=True,
            collate_fn=collate_anchor_batch, drop_last=True,
            pin_memory=(DEVICE.type == "cuda"),
        ),
        "nli": DataLoader(
            ListDataset(nli_triplets), batch_size=CFG.pair_batch_size, shuffle=True,
            collate_fn=collate_triplets, drop_last=True,
        ),
    }
    if criteria_quads:
        loaders["criteria"] = DataLoader(
            ListDataset(criteria_quads), batch_size=CFG.pair_batch_size,
            shuffle=True, collate_fn=collate_quads, drop_last=True,
        )
    if CFG.use_task_train_data:
        loaders["stsb"] = DataLoader(
            ListDataset(stsb_pairs), batch_size=CFG.pair_batch_size, shuffle=True,
            collate_fn=collate_scored_pairs, drop_last=True,
        )
        loaders["qqp"] = DataLoader(
            ListDataset(qqp_triplets), batch_size=CFG.pair_batch_size, shuffle=True,
            collate_fn=collate_triplets, drop_last=True,
        )
        loaders["paws"] = DataLoader(
            ListDataset(paws_triplets), batch_size=CFG.pair_batch_size, shuffle=True,
            collate_fn=collate_triplets, drop_last=True,
        )

    weights = CFG.task_weights
    task_names = [t for t in weights if t in loaders]
    task_probs = np.array([weights[t] for t in task_names], dtype=np.float64)
    task_probs = task_probs / task_probs.sum()
    iterators = {t: infinite_batches(loaders[t]) for t in task_names}

    step_functions = {
        "relation": relation_step,
        "criteria": criteria_step,
        "nli": nli_step,
        "qqp": triplet_step,
        "paws": triplet_step,
        "stsb": cosent_step,
    }

    assert_peft_intact(model, "train_multitask start")

    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer = AdamW(trainable, lr=CFG.lr, weight_decay=CFG.weight_decay)
    scheduler = get_linear_schedule_with_warmup(optimizer, CFG.warmup_steps, CFG.max_steps)

    use_fp16_scaler = DEVICE.type == "cuda" and not torch.cuda.is_bf16_supported()
    try:
        scaler = torch.amp.GradScaler("cuda", enabled=use_fp16_scaler)
    except (AttributeError, TypeError):
        scaler = torch.cuda.amp.GradScaler(enabled=use_fp16_scaler)

                                                                                
    start_step = 1
    best_key = None
    best_state = None
    best_step = None
    evals_since_improvement = 0
    history = []
    running = {t: 0.0 for t in task_names}
    counts = {t: 0 for t in task_names}
    task_rng = random.Random(SEED + 777)

    if RESUME_CHECKPOINT_PATH.exists():
        print(f"Found existing resume checkpoint at {RESUME_CHECKPOINT_PATH} -- resuming training.")
        payload, task_rng = load_resume_checkpoint(
            RESUME_CHECKPOINT_PATH, model, optimizer, scheduler, scaler
        )
        start_step = payload["step"] + 1
        running = payload["running"]
        counts = payload["counts"]
        history = payload["history"]
        best_key = payload["best_key"]
        best_state = payload["best_state"]
        best_step = payload["best_step"]
                                                                      
                                                  
        evals_since_improvement = payload.get("evals_since_improvement", 0)
        assert_peft_intact(model, "train_multitask after resume")
        if start_step > CFG.max_steps:
            print(f"Resume checkpoint is already at step {payload['step']} "
                  f">= CFG.max_steps ({CFG.max_steps}); nothing left to train.")
    else:
        print("No resume checkpoint found -- starting training from step 1.")

    def _snapshot(step_now, why):
        """Single place that writes the resume checkpoint, so every trigger
        (interval / eval / completion / crash / Ctrl-C) saves identical state."""
        save_resume_checkpoint(
            RESUME_CHECKPOINT_PATH, step_now, model, optimizer, scheduler, scaler,
            task_rng, running, counts, history, best_key, best_state, best_step,
            evals_since_improvement,
        )
        print(f"[checkpoint] trigger: {why}")

    progress = tqdm(
        range(start_step, CFG.max_steps + 1),
        initial=start_step - 1, total=CFG.max_steps, desc="multitask training",
    )
    step = start_step - 1
    try:
      for step in progress:
          model.train()
          task = task_rng.choices(task_names, weights=task_probs, k=1)[0]
          batch = next(iterators[task])

          loss = step_functions[task](model, batch)
          optimizer.zero_grad(set_to_none=True)
          if scaler.is_enabled():
              scaler.scale(loss).backward()
              scaler.unscale_(optimizer)
              torch.nn.utils.clip_grad_norm_(trainable, CFG.max_grad_norm)
              scaler.step(optimizer)
              scaler.update()
          else:
              loss.backward()
              torch.nn.utils.clip_grad_norm_(trainable, CFG.max_grad_norm)
              optimizer.step()
          scheduler.step()

          running[task] += loss.detach().item()
          counts[task] += 1
          if step % 50 == 0:
              progress.set_postfix({
                  t: f"{running[t] / max(1, counts[t]):.3f}" for t in task_names
              })

          if step % CFG.eval_every == 0 or step == CFG.max_steps:
              metrics = run_checkpoint_eval(model, step)  # step -> rotates the criteria probe
              key, mean_delta, deltas, balanced_delta, normalized = selection_key(
                  metrics, BASELINE
              )
              crit = metrics["criteria"]
              history.append({
                  "step": step,
                  "alignment": metrics["alignment"],
                  "c3_margin": crit["c3_margin"],
                  "c4_margin": crit["c4_margin"],
                  "c3_pass": crit["c3_pass"],
                  "c4_pass": crit["c4_pass"],
                  "stsb": metrics["stsb"],
                  "mrpc_f1": metrics["mrpc_f1"],
                  "qqp_f1": metrics["qqp_f1"],
                  "paws_acc": metrics["paws_acc"],
                  "polarity": metrics["polarity"],
                  "mean_delta": mean_delta,
                  "balanced_delta": balanced_delta,
                  "criteria_ok": key[1] == 1,
                  "both_objectives": key[0] == 1,
              })
              print_alignment(f"step {step} dev", metrics["alignment_full"])
              print(
                  f"STS-B={metrics['stsb']:.4f} ({deltas[0]:+.4f}) | "
                  f"MRPC-F1={metrics['mrpc_f1']:.4f} ({deltas[1]:+.4f}) | "
                  f"QQP-F1={metrics['qqp_f1']:.4f} ({deltas[2]:+.4f}) | "
                  f"PAWS={metrics['paws_acc']:.4f} ({deltas[3]:+.4f})"
              )
              crit = metrics["criteria"]
              print(
                  f"C3 margin={crit['c3_margin']:+.4f} "
                  f"pass@eps={crit['c3_pass']:.1%} "
                  f"(para={crit['avg_para_sim']:.3f} ant={crit['avg_ant_sim']:.3f})"
              )
              print(
                  f"C4 margin={crit['c4_margin']:+.4f} "
                  f"pass@eps={crit['c4_pass']:.1%} "
                  f"(para={crit['avg_para_sim']:.3f} jum={crit['avg_jum_sim']:.3f})"
              )
              geo = metrics["geometry"]
              print(
                  f"geometry (abs sim by rank class): "
                  f"equiv={geo['equiv_sim']:.3f} opposite={geo['opposite_sim']:.3f} "
                  f"jumbled={geo['jumbled_sim']:.3f} distinct={geo['distinct_sim']:.3f} | "
                  # f"equiv-opposite={geo['equiv_minus_opposite']:+.3f} "
                  # f"{'OK' if geo['equiv_beats_opposite'] else 'TOO NARROW'} | "
                  # f"opposite-jumbled={geo['opposite_minus_jumbled']:+.3f} | "
                  # f"full range={geo['equiv_minus_distinct']:+.3f} | "
                  # f"ordering {'OK' if geo['ordering_ok'] else 'VIOLATED'}"
              )
              _worst = min(normalized, key=normalized.get)
              _pol_ok = metrics["polarity"] <= CFG.polarity_ceiling
              print(f"balanced delta (worst normalized metric): {balanced_delta:+.4f} "
                    f"[{_worst}] | polarity (equiv-distinct): {metrics['polarity']:+.3f} "
                    f"(ceiling {CFG.polarity_ceiling:.2f} "
                    f"{'OK' if _pol_ok else 'EXCEEDED -- checkpoint deprioritized'})")
              print(f"mean delta over baseline: {mean_delta:+.4f} | "
                    f"criteria (C3&C4 >50%) {'MET' if key[1] else 'NOT MET'}")
              if best_key is None or key > best_key:
                  best_key = key
                  best_step = step
                  best_state = {
                      k: v.detach().cpu().clone()
                      for k, v in get_peft_model_state_dict(model[0].auto_model).items()
                  }
                  evals_since_improvement = 0
                  print("Saved new best checkpoint.")
              else:
                  evals_since_improvement += 1
                  print(f"No improvement for {evals_since_improvement}/{CFG.patience} "
                        f"eval(s) (best so far: step {best_step}).")

          if (step % CFG.resume_checkpoint_every == 0
                  or step % CFG.eval_every == 0
                  or step == CFG.max_steps):
              _snapshot(step, "interval/eval/completion")

          if (step % CFG.eval_every == 0
                  and CFG.patience is not None
                  and evals_since_improvement >= CFG.patience):
              print(f"\n[early stop] no improvement for {evals_since_improvement} "
                    f"consecutive evals (patience={CFG.patience}) -- "
                    f"stopping at step {step} (best was step {best_step}).")
              break

    except KeyboardInterrupt:
        print(f"\n[checkpoint] interrupted at step {step} -- saving before exit.")
        _snapshot(step, "KeyboardInterrupt")
        raise
    except Exception:
        print(f"\n[checkpoint] exception at step {step} -- saving before re-raising.")
        _snapshot(step, "exception")
        raise

    if best_state is None:
        raise RuntimeError("Training finished without a saved checkpoint.")
    if not any("lora_B" in k for k in best_state):
        raise RuntimeError(
            "best_state has no lora_B tensors - LoRA never trained. "
            f"keys sample: {list(best_state)[:5]}"
        )
    set_peft_model_state_dict(model[0].auto_model, best_state) 
    model.eval()
    print("Restored best checkpoint from step:", best_step)
    return model, best_state, pd.DataFrame(history)

model, best_state, history = train_multitask()
display(history)
Path(CFG.output_dir).mkdir(parents=True, exist_ok=True)
torch.save(best_state, Path(CFG.output_dir) / "best_lora_state.pt")
print("Saved best_lora_state.pt |", 
      len(best_state), "tensors |",
      sum(1 for k in best_state if "lora_B" in k), "lora_B tensors")  


building sentence-transformers/all-mpnet-base-v2 (first call downloads the checkpoint; later calls hit the cache)...
model ready (mean pooling).
Wrapped model type: PeftModelForFeatureExtraction
trainable params: 884,736 || all params: 110,371,200 || trainable%: 0.8016
LoRA parameter tensors found: 48
LoRA tensors visible to the optimizer: 48
Gradient checkpointing unavailable (MPNetModel does not support gradient checkpointing.); continuing without it.
Usable training anchors: 15631
No resume checkpoint found -- starting training from step 1.


multitask training:   0%|          | 0/12000 [00:00<?, ?it/s]

[checkpoint] step 250: resume checkpoint saved -> /kaggle/working/align_mpnet_mtl/resume_checkpoint.pt
[checkpoint] trigger: interval/eval/completion

step 500 dev
macro alignment: 0.7245
overall pairwise: 0.7262
  -- criteria relations (in macro) --
  paraphrase    : 0.7661
  synonym       : 0.9559
  entailment    : 0.5372
  contradiction : 0.7838
  antonym       : 0.6044
  negation      : 0.6894
  jumbled       : 0.4671
  distinction   : 0.9921
  -- other relations (reported only) --
STS-B=0.8864 (+0.0054) | MRPC-F1=0.8352 (+0.0008) | QQP-F1=0.7624 (+0.0038) | PAWS=0.6300 (+0.0027)
C3 margin=+0.0056 pass@eps=31.4% (para=0.906 ant=0.901)
C4 margin=-0.0078 pass@eps=20.8% (para=0.906 jum=0.914)
geometry (abs sim by rank class): equiv=0.871 opposite=0.651 jumbled=0.912 distinct=0.023 | 
balanced delta (worst normalized metric): +0.0046 [mrpc_f1] | polarity (equiv-distinct): +0.848 (ceiling 0.78 EXCEEDED -- checkpoint deprioritized)
mean delta over baseline: +0.0031 | criteria (C3&C4 >50%

,step,alignment,c3_margin,c4_margin,c3_pass,c4_pass,stsb,mrpc_f1,qqp_f1,paws_acc,polarity,mean_delta,balanced_delta,criteria_ok,both_objectives
0,500,0.724496,0.005574,-0.007773,0.314,0.208,0.886407,0.835225,0.762418,0.630000,0.848104,0.003142,0.004577,False,False
1,1000,0.748756,0.039298,0.011039,0.440,0.254,0.886208,0.836430,0.760664,0.632000,0.798138,0.003455,0.008297,False,False
2,1500,0.759731,0.051434,0.024808,0.496,0.344,0.890153,0.831637,0.758288,0.626667,0.764897,0.001316,-0.017095,False,False
3,2000,0.797100,0.070643,0.087783,0.558,0.656,0.890096,0.838739,0.761010,0.626667,0.728687,0.003758,-0.001789,True,True
4,2500,0.822797,0.086958,0.150364,0.648,0.800,0.892374,0.840382,0.757831,0.630667,0.707110,0.004943,-0.003439,True,True
5,3000,0.840061,0.089153,0.193180,0.656,0.836,0.892723,0.838798,0.761628,0.634000,0.704603,0.006417,0.012293,True,True
6,3500,0.849701,0.086867,0.223537,0.618,0.876,0.893777,0.839562,0.762071,0.634667,0.700907,0.007149,0.014129,True,True
7,4000,0.861880,0.096473,0.246091,0.658,0.878,0.892424,0.835698,0.758750,0.636000,0.673144,0.005348,0.000368,True,True
8,4500,0.870345,0.102486,0.295176,0.672,0.912,0.892692,0.838121,0.762353,0.642667,0.688893,0.008588,0.015297,True,True
9,5000,0.868960,0.104480,0.286292,0.680,0.892,0.892666,0.838416,0.762985,0.644000,0.707179,0.009146,0.017916,True,True


Saved best_lora_state.pt | 48 tensors | 24 lora_B tensors


In [23]:
def make_encoder_from_model(st_model):
    def encode(sentences, batch_size=64): 
        return st_model.encode(list(sentences), batch_size=batch_size, convert_to_numpy=True,
                                normalize_embeddings=True, show_progress_bar=False)
    return encode

In [24]:
CHECKPOINT_PATH = Path(CFG.output_dir) / "best_lora_state.pt"
assert CHECKPOINT_PATH.exists(), f"Checkpoint not found at {CHECKPOINT_PATH}" 

seed_everything(SEED)
model = build_lora_model()
assert_peft_intact(model, "resume: after build_lora_model")

best_state = torch.load(CHECKPOINT_PATH, map_location="cpu")
best_state = {k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in best_state.items()}
if not any("lora_B" in k for k in best_state):
    raise RuntimeError("Loaded state has no lora_B tensors -- wrong checkpoint file?")

set_peft_model_state_dict(model[0].auto_model, best_state)
model.eval()
assert_peft_intact(model, "resume: after loading checkpoint")

print(f"Loaded checkpoint: {len(best_state)} tensors "
      f"({sum(1 for k in best_state if 'lora_B' in k)} lora_B tensors)")

building sentence-transformers/all-mpnet-base-v2 (first call downloads the checkpoint; later calls hit the cache)...
model ready (mean pooling).
Wrapped model type: PeftModelForFeatureExtraction
trainable params: 884,736 || all params: 110,371,200 || trainable%: 0.8016
LoRA parameter tensors found: 48
LoRA tensors visible to the optimizer: 48
Gradient checkpointing unavailable (MPNetModel does not support gradient checkpointing.); continuing without it.
Loaded checkpoint: 48 tensors (24 lora_B tensors)


In [25]:
assert_peft_intact(model, "before WiSE-FT")
wise_rows = []
for lam in CFG.wise_ft_lambdas:
    set_peft_model_state_dict(model[0].auto_model, scaled_lora_state(best_state, lam))
    model.eval()
    metrics = run_checkpoint_eval(model)
    key, mean_delta, deltas, balanced_delta, normalized = selection_key(metrics, BASELINE)
    crit = metrics["criteria"]
    no_regression = all(d >= -CFG.delta_tolerance for d in deltas)
    wise_rows.append({
        "lambda": lam,
        "alignment": metrics["alignment"],
        "c3_margin": crit["c3_margin"],
        "c4_margin": crit["c4_margin"],
        "c3_pass": crit["c3_pass"],
        "c4_pass": crit["c4_pass"],
        "criteria_pass": metrics["criteria_pass"],
        "stsb": metrics["stsb"],
        "mrpc_f1": metrics["mrpc_f1"],
        "qqp_f1": metrics["qqp_f1"],
        "paws_acc": metrics["paws_acc"],
        "polarity": metrics["polarity"],
        "mean_delta": mean_delta,
        "balanced_delta": balanced_delta,
        "no_regression": no_regression,
        "criteria_ok": key[1] == 1,
    })
    print(f"lambda={lam}: C3pass={crit['c3_pass']:.1%} C4pass={crit['c4_pass']:.1%} "
          f"stsb={metrics['stsb']:.4f} balanced_delta={balanced_delta:+.4f} "
          f"polarity={metrics['polarity']:+.3f}")

wise_table = pd.DataFrame(wise_rows)
wise_table["criteria_ok"] = (wise_table["c3_pass"] > 0.5) & (wise_table["c4_pass"] > 0.5)                                                                     
wise_table["polarity_ok"] = wise_table["polarity"] <= CFG.polarity_ceiling
_qualified = wise_table[
    wise_table["criteria_ok"] & wise_table["no_regression"] & wise_table["polarity_ok"]
]
if _qualified.empty:                                           
    _relaxed = wise_table[wise_table["criteria_ok"] & wise_table["no_regression"]]
    if not _relaxed.empty:
        print(f"No lambda is under the polarity ceiling ({CFG.polarity_ceiling:.2f}); "
              f"relaxing that gate only. Lowest polarity available: "
              f"{wise_table['polarity'].min():.3f}")
        _qualified = _relaxed

def _minmax(series):
    lo, hi = series.min(), series.max()
    if hi - lo < 1e-12:                                            
        return pd.Series(0.5, index=series.index)
    return (series - lo) / (hi - lo)

if not _qualified.empty:
    print(f"{len(_qualified)} lambda(s) satisfy C3/C4 (>50%) and no_regression -- "
          f"selecting the one that maximizes balanced_delta among them "
          f"(criteria_pass and polarity earn nothing further once satisfied).")
    best_row = _qualified.sort_values("balanced_delta", ascending=False).iloc[0]
    wise_table["composite_score"] = float("nan")                       
else:
    print("No lambda satisfies C3/C4 (>50%) and no_regression -- "
          "falling back to the criteria/downstream composite score to steer toward satisfying them.")
    wise_table["criteria_norm"] = _minmax(wise_table["criteria_pass"])
    wise_table["downstream_norm"] = _minmax(wise_table["balanced_delta"])
    _w = CFG.wise_ft_criteria_weight
    wise_table["composite_score"] = (
        _w * wise_table["criteria_norm"] + (1 - _w) * wise_table["downstream_norm"]
    )
    _candidates = wise_table[wise_table["no_regression"]]
    if _candidates.empty:
        print("No lambda satisfies no_regression either -- selecting from the full sweep instead.")
        _candidates = wise_table
    best_row = _candidates.sort_values("composite_score", ascending=False).iloc[0]

display(wise_table)

best_lambda = float(best_row["lambda"])
print(f"\nSelected lambda: {best_lambda} "
      f"(criteria_ok={bool(best_row['criteria_ok'])}, "
      f"C3pass={best_row['c3_pass']:.1%}, C4pass={best_row['c4_pass']:.1%}, "
      f"balanced_delta={best_row['balanced_delta']:+.4f}, "
      f"stsb={best_row['stsb']:.4f}, polarity={best_row['polarity']:+.3f} "
      f"(ceiling {CFG.polarity_ceiling:.2f}), "
      f"no_regression={bool(best_row['no_regression'])})")

if wise_table["stsb"].nunique() == 1 and len(wise_table) > 1:
    raise RuntimeError(
        "All lambdas gave the SAME STS-B score. The LoRA adapter is not active "
        "in the forward pass. Do not trust any downstream metric."
    )

set_peft_model_state_dict(model[0].auto_model, scaled_lora_state(best_state, best_lambda))
model.eval()
assert_peft_intact(model, "after WiSE-FT")
torch.save({"best_lambda": best_lambda}, Path(CFG.output_dir) / "best_lambda.pt")


lambda=0.3: C3pass=42.0% C4pass=24.2% stsb=0.8876 balanced_delta=+0.0161 polarity=+0.826
lambda=0.4: C3pass=45.3% C4pass=30.4% stsb=0.8895 balanced_delta=+0.0161 polarity=+0.815
lambda=0.5: C3pass=51.0% C4pass=40.0% stsb=0.8911 balanced_delta=+0.0054 polarity=+0.802
lambda=0.6: C3pass=55.2% C4pass=51.2% stsb=0.8923 balanced_delta=+0.0036 polarity=+0.785
lambda=0.75: C3pass=61.6% C4pass=72.5% stsb=0.8929 balanced_delta=+0.0235 polarity=+0.756
lambda=0.85: C3pass=65.0% C4pass=82.3% stsb=0.8929 balanced_delta=-0.0223 polarity=+0.733
lambda=1.0: C3pass=70.5% C4pass=91.8% stsb=0.8920 balanced_delta=+0.0355 polarity=+0.695
2 lambda(s) satisfy C3/C4 (>50%) and no_regression -- selecting the one that maximizes balanced_delta among them (criteria_pass and polarity earn nothing further once satisfied).


,lambda,alignment,c3_margin,c4_margin,c3_pass,c4_pass,criteria_pass,stsb,mrpc_f1,qqp_f1,paws_acc,polarity,mean_delta,balanced_delta,no_regression,criteria_ok,polarity_ok,composite_score
0,0.30,0.730194,0.025002,-0.004082,0.420,0.242,0.3310,0.887601,0.840000,0.763913,0.633333,0.826452,0.005842,0.016100,True,False,False,NaN
1,0.40,0.740292,0.036564,0.008652,0.453,0.304,0.3785,0.889546,0.842617,0.767700,0.633333,0.815356,0.007929,0.016100,True,False,False,NaN
2,0.50,0.753611,0.048531,0.028686,0.510,0.400,0.4550,0.891131,0.842667,0.764286,0.629333,0.801552,0.006484,0.005367,True,False,False,NaN
3,0.60,0.772403,0.060818,0.060049,0.552,0.512,0.5320,0.892258,0.840747,0.759540,0.632667,0.785067,0.005933,0.003640,True,True,False,NaN
4,0.75,0.812508,0.079608,0.136658,0.616,0.725,0.6705,0.892868,0.840970,0.764331,0.638667,0.755521,0.008839,0.023494,True,True,True,NaN
5,0.85,0.840156,0.092167,0.208580,0.650,0.823,0.7365,0.892894,0.830783,0.763396,0.637333,0.732849,0.005731,-0.022253,False,True,True,NaN
6,1.00,0.881515,0.110335,0.334268,0.705,0.918,0.8115,0.892014,0.841392,0.767236,0.646667,0.694905,0.011457,0.035531,True,True,True,NaN



Selected lambda: 1.0 (criteria_ok=True, C3pass=70.5%, C4pass=91.8%, balanced_delta=+0.0355, stsb=0.8920, polarity=+0.695 (ceiling 0.78), no_regression=True)


In [26]:
history = pd.DataFrame()

In [27]:
_probe = ["a quick sanity probe sentence", "an unrelated second sentence"]
_v_loaded = model.encode(_probe, normalize_embeddings=True)
_zeroed = {k: (torch.zeros_like(v) if "lora_B" in k else v) for k, v in best_state.items()}
set_peft_model_state_dict(model[0].auto_model, _zeroed)
_v_zero = model.encode(_probe, normalize_embeddings=True)
                                                                             
                                                                               
_restore_state = (scaled_lora_state(best_state, best_lambda)
                  if "best_lambda" in dir() else best_state)
set_peft_model_state_dict(model[0].auto_model, _restore_state)
model.eval()
_delta = float(np.abs(_v_loaded - _v_zero).mean())
print(f"embedding delta vs. lora_B=0: {_delta:.8f}")
if _delta < 1e-6:
    raise RuntimeError("Loaded checkpoint is inert -- adapter isn't contributing anything.")

history = pd.DataFrame()
best_step = "resumed-from-checkpoint"

embedding delta vs. lora_B=0: 0.01380993


In [28]:
from copy import deepcopy
merged_directory = Path(CFG.output_dir) / "sentence_transformer_merged"
_merge_model = build_lora_model()
_state_to_merge = (scaled_lora_state(best_state, best_lambda)
                   if "best_lambda" in dir() else best_state)
set_peft_model_state_dict(_merge_model[0].auto_model, _state_to_merge)
_merge_model[0].auto_model = _merge_model[0].auto_model.merge_and_unload()
_merge_model.max_seq_length = BASE_MODEL_MAX_SEQ_LEN
_merge_model.eval()
_merge_model.save(str(merged_directory))
print(f"Merged model saved -> {merged_directory}")                                                                            
_probe_sents = ["a quick sanity probe sentence", "an unrelated second sentence"]
_base_probe = build_base_model()
_v_base = _base_probe.encode(_probe_sents, normalize_embeddings=True)
_v_merged = _merge_model.encode(_probe_sents, normalize_embeddings=True)
_merge_delta = float(np.abs(_v_base - _v_merged).mean())
print(f"merged-vs-base embedding delta: {_merge_delta:.8f}")
if _merge_delta < 1e-6:
    raise RuntimeError(
        "Merged model is identical to the base model -- the adapter was not "
        "merged. Every 'ours' number downstream would be the baseline."
    )
print("Merge verified: the adapter is baked into the merged weights.")

del _merge_model, _base_probe
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


building sentence-transformers/all-mpnet-base-v2 (first call downloads the checkpoint; later calls hit the cache)...
model ready (mean pooling).
Wrapped model type: PeftModelForFeatureExtraction
trainable params: 884,736 || all params: 110,371,200 || trainable%: 0.8016
LoRA parameter tensors found: 48
LoRA tensors visible to the optimizer: 48
Gradient checkpointing unavailable (MPNetModel does not support gradient checkpointing.); continuing without it.
Merged model saved -> /kaggle/working/align_mpnet_mtl/sentence_transformer_merged
building sentence-transformers/all-mpnet-base-v2 (first call downloads the checkpoint; later calls hit the cache)...
model ready (mean pooling).
merged-vs-base embedding delta: 0.01380993
Merge verified: the adapter is baked into the merged weights.


# Clean Final Evaluation

In [29]:
from datasets import load_dataset
import numpy as np

print("Loading downstream evaluation datasets...")
stsb_ds = load_dataset("sentence-transformers/stsb")["test"]

mrpc_hf = load_dataset("glue", "mrpc")
mrpc_train_ds = mrpc_hf["train"]
mrpc_test_ds = mrpc_hf["validation"]

paws_hf = load_dataset("google-research-datasets/paws", "labeled_final")
paws_train_ds = paws_hf["train"]
paws_test_ds = paws_hf["test"]

qqp_hf = load_dataset("glue", "qqp")
qqp_train_ds = qqp_hf["train"].rename_columns(
    {"question1": "sentence1", "question2": "sentence2"}
)
qqp_test_ds = qqp_hf["validation"].rename_columns(
    {"question1": "sentence1", "question2": "sentence2"}
)

Loading downstream evaluation datasets...


README.md: 0.00B [00:00, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/471k [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/142k [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/108k [00:00<?, ?B/s]

...

validation-00000-of-00001.parquet:   0%|          | 0.00/3.73M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/36.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/363846 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/40430 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/390965 [00:00<?, ? examples/s]

In [30]:
from datasets import load_dataset
import numpy as np

print("Loading downstream evaluation datasets...")
stsb_ds = load_dataset("sentence-transformers/stsb")["test"]

mrpc_hf = load_dataset("glue", "mrpc")
mrpc_train_ds = mrpc_hf["train"]
mrpc_test_ds = mrpc_hf["validation"]

paws_hf = load_dataset("google-research-datasets/paws", "labeled_final")
paws_train_ds = paws_hf["train"]
paws_test_ds = paws_hf["test"]

qqp_hf = load_dataset("glue", "qqp")
qqp_train_ds = qqp_hf["train"].rename_columns(
    {"question1": "sentence1", "question2": "sentence2"}
)
qqp_test_ds = qqp_hf["validation"].rename_columns(
    {"question1": "sentence1", "question2": "sentence2"}
)

print(f"STS-B test bed: {len(stsb_ds):,}")
print(f"MRPC:  train={len(mrpc_train_ds):,}  test bed={len(mrpc_test_ds):,}")
print(f"PAWS:  train={len(paws_train_ds):,}  test bed={len(paws_test_ds):,}")
print(f"QQP:   train={len(qqp_train_ds):,}  test bed (validation)={len(qqp_test_ds):,}")

Loading downstream evaluation datasets...
STS-B test bed: 1,379
MRPC:  train=3,668  test bed=408
PAWS:  train=49,401  test bed=8,000
QQP:   train=363,846  test bed (validation)=40,430


In [31]:
def exclude_trained_pairs(train_ds, used_pairs_varname):
    try:
        used_pairs = {(t["anchor"], t["positive"]) for t in globals()[used_pairs_varname]}
    except KeyError:
        print(f"  {used_pairs_varname} not in memory -- using the full train split for tuning.")
        return train_ds
    s1, s2 = list(train_ds["sentence1"]), list(train_ds["sentence2"])
    eligible = [i for i in range(len(train_ds)) if (s1[i], s2[i]) not in used_pairs]
    print(f"  excluding {len(train_ds) - len(eligible):,} rows already used for training "
          f"({len(eligible):,} remain)")
    return train_ds.select(eligible)

print("QQP tuning pool:")
qqp_train_ds = exclude_trained_pairs(qqp_train_ds, "qqp_triplets")
print("PAWS tuning pool:")
paws_train_ds = exclude_trained_pairs(paws_train_ds, "paws_triplets")

QQP tuning pool:
  excluding 60,000 rows already used for training (303,846 remain)
PAWS tuning pool:
  excluding 18,437 rows already used for training (30,964 remain)


In [55]:
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def make_st_encoder(model_id_or_path, prefix=""):
    st_model = SentenceTransformer(model_id_or_path, device=DEVICE)
    def encode(sentences, batch_size=64):
        texts = [prefix + s for s in sentences] if prefix else list(sentences)
        return st_model.encode(texts, batch_size=batch_size, convert_to_numpy=True,
                                normalize_embeddings=True, show_progress_bar=False)
    return encode

In [56]:
from scipy.stats import spearmanr

def eval_stsb(encode_fn, ds):
    e1 = encode_fn(list(ds["sentence1"]))
    e2 = encode_fn(list(ds["sentence2"]))
    cosine = np.sum(e1 * e2, axis=1)
    scores = np.asarray(ds["score"], dtype=np.float32)
    return float(spearmanr(cosine, scores).correlation)

def find_best_threshold(encode_fn, tune_ds):
    s1, s2 = list(tune_ds["sentence1"]), list(tune_ds["sentence2"])
    labels = np.asarray(tune_ds["label"], dtype=np.int64)
    e1, e2 = encode_fn(s1), encode_fn(s2)
    cosine = np.sum(e1 * e2, axis=1)
    def f1_at(t):
        preds = (cosine >= t).astype(int)
        tp = np.sum((preds == 1) & (labels == 1))
        fp = np.sum((preds == 1) & (labels == 0))
        fn = np.sum((preds == 0) & (labels == 1))
        return tp / (tp + 0.5 * (fp + fn) + 1e-10)
    return max(np.linspace(cosine.min(), cosine.max(), 200), key=f1_at)

def evaluate_on_test_bed(encode_fn, test_ds, threshold):
    s1, s2 = list(test_ds["sentence1"]), list(test_ds["sentence2"])
    labels = np.asarray(test_ds["label"], dtype=np.int64)
    e1, e2 = encode_fn(s1), encode_fn(s2)
    cosine = np.sum(e1 * e2, axis=1)
    preds = (cosine >= threshold).astype(int)
    tp = np.sum((preds == 1) & (labels == 1))
    fp = np.sum((preds == 1) & (labels == 0))
    fn = np.sum((preds == 0) & (labels == 1))
    f1 = tp / (tp + 0.5 * (fp + fn) + 1e-10)
    acc = float(np.mean(preds == labels))
    return {"f1": float(f1), "acc": acc, "threshold": float(threshold)}

def eval_pair_classification(encode_fn, train_ds, test_ds):
    threshold = find_best_threshold(encode_fn, train_ds)
    return evaluate_on_test_bed(encode_fn, test_ds, threshold)

In [57]:
# Use the MERGED model (saved at BASE_MODEL_MAX_SEQ_LEN, see Cell 27),
# not the pre-merge `model` object -- that one's max_seq_length is still
# CFG.max_seq_length (128, set in build_base_model() and never updated).
# Evaluating "Our model" at 128 tokens here while the MTEB section
# evaluates it at BASE_MODEL_MAX_SEQ_LEN (512) via merged_directory was an
# inconsistency: this table's numbers weren't using the model's full
# context length.
_eval_model = SentenceTransformer(str(merged_directory), device=str(DEVICE))
_eval_model.eval()
print(f"'Our model' eval max_seq_length: {_eval_model.max_seq_length} "
      f"(was silently 128 before this fix)")
Align_mpnet = make_encoder_from_model(_eval_model)

'Our model' eval max_seq_length: 512 (was silently 128 before this fix)


In [58]:
import pandas as pd, gc

MODELS = {
    "Our model":        {"type": "prebuilt", "encode_fn": Align_mpnet},
}

In [59]:
def eval_pair_classification(encode_fn, train_ds, test_ds, task_name=""):
    print(f"  [{task_name}] tuning threshold on {len(train_ds):,} pairs...")
    threshold = find_best_threshold(encode_fn, train_ds)
    print(f"  [{task_name}] threshold={threshold:.4f}, evaluating on {len(test_ds):,} pairs...")
    return evaluate_on_test_bed(encode_fn, test_ds, threshold)

In [60]:
def subsample_ds(ds, n, seed=42):
    if len(ds) <= n:
        return ds
    rng = np.random.RandomState(seed)
    idx = rng.choice(len(ds), size=n, replace=False)
    return ds.select(idx) if hasattr(ds, "select") else {k: [v[i] for i in idx] for k, v in ds.items()}

qqp_train_ds = subsample_ds(qqp_train_ds, 10000)
paws_train_ds = subsample_ds(paws_train_ds, 10000)
print(f"QQP train (subsampled): {len(qqp_train_ds):,}")
print(f"PAWS train (subsampled): {len(paws_train_ds):,}")
from sklearn.model_selection import train_test_split as _qqp_tts

_qqp_labels = np.asarray(qqp_train_ds["label"])
_qqp_idx = np.arange(len(qqp_train_ds))
_qqp_train_idx, _qqp_val_idx = _qqp_tts(
    _qqp_idx, test_size=0.2, random_state=SEED, stratify=_qqp_labels
)
qqp_val_ds = qqp_train_ds.select(sorted(_qqp_val_idx))
qqp_train_ds = qqp_train_ds.select(sorted(_qqp_train_idx))
print(f"QQP tune/val split: {len(qqp_train_ds):,} train (80%, used for threshold tuning) / "
      f"{len(qqp_val_ds):,} val (20%, held out -- NOT used anywhere in testing below)")

QQP train (subsampled): 6,400
PAWS train (subsampled): 10,000
QQP tune/val split: 5,120 train (80%, used for threshold tuning) / 1,280 val (20%, held out -- NOT used anywhere in testing below)


In [61]:
rows = []
for model_name, cfg in MODELS.items():
    print(f"\n=== {model_name} ===")
    if cfg["type"] == "st":
        encode_fn = make_st_encoder(cfg["id"], prefix=cfg.get("prefix", ""))
    elif cfg["type"] == "prebuilt":
        encode_fn = cfg["encode_fn"]

    stsb_score = eval_stsb(encode_fn, stsb_ds)
    mrpc_result = eval_pair_classification(encode_fn, mrpc_train_ds, mrpc_test_ds, task_name="MRPC")
    qqp_result = eval_pair_classification(encode_fn, qqp_train_ds, qqp_test_ds, task_name="QQP")
    paws_result = eval_pair_classification(encode_fn, paws_train_ds, paws_test_ds, task_name="PAWS")

    rows.append({
        "model": model_name, "STS-B (Spearman)": stsb_score,
        "MRPC F1": mrpc_result["f1"], "MRPC Acc": mrpc_result["acc"],
        "QQP F1": qqp_result["f1"], "QQP Acc": qqp_result["acc"],
        "PAWS F1": paws_result["f1"], "PAWS Acc": paws_result["acc"],
    })
    print(f"STS-B={stsb_score:.4f}  MRPC-F1={mrpc_result['f1']:.4f}  "
          f"QQP-F1={qqp_result['f1']:.4f}  PAWS-F1={paws_result['f1']:.4f}")

results_df = pd.DataFrame(rows).set_index("model")
display(results_df.round(4))


=== Our model ===
  [MRPC] tuning threshold on 3,668 pairs...
  [MRPC] threshold=0.7655, evaluating on 408 pairs...
  [QQP] tuning threshold on 5,120 pairs...
  [QQP] threshold=0.8324, evaluating on 40,430 pairs...
  [PAWS] tuning threshold on 10,000 pairs...
  [PAWS] threshold=0.9888, evaluating on 8,000 pairs...
STS-B=0.8500  MRPC-F1=0.8387  QQP-F1=0.7499  PAWS-F1=0.5529


,STS-B (Spearman),MRPC F1,MRPC Acc,QQP F1,QQP Acc,PAWS F1,PAWS Acc
model,,,,,,,
Our model,0.85,0.8387,0.7549,0.7499,0.8062,0.5529,0.6446


In [62]:
import inspect
if not hasattr(inspect, "getargspec"):
    inspect.getargspec = inspect.getfullargspec

In [63]:
# Verify the SAVED config, not an in-memory object. This is what evaluation reads.
from sentence_transformers import SentenceTransformer
import json as _json

_cfg = Path(merged_directory) / "sentence_bert_config.json"
print("sentence_bert_config.json:", _json.load(open(_cfg)))
print("reloaded max_seq_length  :", SentenceTransformer(str(merged_directory)).max_seq_length)
print("training length was      :", CFG.max_seq_length, "(128 is correct for training)")
assert SentenceTransformer(str(merged_directory)).max_seq_length == BASE_MODEL_MAX_SEQ_LEN, \
    (f"saved model is not at {BASE_MODEL_MAX_SEQ_LEN} -- rerun the merge cell, "
     "then clear the mteb cache")

sentence_bert_config.json: {'max_seq_length': 512, 'do_lower_case': False}
reloaded max_seq_length  : 512
training length was      : 128 (128 is correct for training)


In [64]:
_state_to_merge = (scaled_lora_state(best_state, best_lambda)
                   if "best_lambda" in dir() else best_state)

In [65]:
from datasets import load_dataset

TASK_CANDIDATES = {
    "MR":   ["SetFit/mr", "rotten_tomatoes", "cornell-movie-review-data/rotten_tomatoes"],
    "CR":   ["SetFit/CR", "SetFit/cr"],
    "SUBJ": ["SetFit/subj"],
    "MPQA": ["SetFit/mpqa"],
    "SST2": ["SetFit/sst2", "glue,sst2"],   # special-cased below (glue needs a config arg)
    "TREC": ["SetFit/trec", "trec", "CogComp/trec"],
}

def load_first_available(task_name, candidates):
    for cand in candidates:
        try:
            if cand == "glue,sst2":
                ds = load_dataset("glue", "sst2")
            elif cand == "trec":
                ds = load_dataset("trec", trust_remote_code=False)
            else:
                ds = load_dataset(cand)
            print(f"{task_name}: loaded from '{cand}'  columns={ds[list(ds.keys())[0]].column_names}  "
                  f"splits={list(ds.keys())}")
            return cand, ds
        except Exception as exc:
            print(f"{task_name}: '{cand}' failed ({type(exc).__name__}: {exc})")
    print(f"{task_name}: NO candidate worked -- skipping this task.")
    return None, None

loaded_tasks = {}
for task_name, candidates in TASK_CANDIDATES.items():
    source, ds = load_first_available(task_name, candidates)
    if ds is not None:
        loaded_tasks[task_name] = ds

MR: 'SetFit/mr' failed (DatasetNotFoundError: Dataset 'SetFit/mr' doesn't exist on the Hub or cannot be accessed.)


Repo card metadata block was not found. Setting CardData to empty.


MR: loaded from 'rotten_tomatoes'  columns=['text', 'label']  splits=['train', 'validation', 'test']


Repo card metadata block was not found. Setting CardData to empty.


CR: loaded from 'SetFit/CR'  columns=['text', 'label', 'label_text']  splits=['train', 'test']
SUBJ: loaded from 'SetFit/subj'  columns=['text', 'label', 'label_text']  splits=['train', 'test']
MPQA: 'SetFit/mpqa' failed (DatasetNotFoundError: Dataset 'SetFit/mpqa' doesn't exist on the Hub or cannot be accessed.)
MPQA: NO candidate worked -- skipping this task.


Repo card metadata block was not found. Setting CardData to empty.


SST2: loaded from 'SetFit/sst2'  columns=['text', 'label', 'label_text']  splits=['train', 'validation', 'test']
TREC: 'SetFit/trec' failed (DatasetNotFoundError: Dataset 'SetFit/trec' doesn't exist on the Hub or cannot be accessed.)
TREC: 'trec' failed (RuntimeError: Dataset scripts are no longer supported, but found trec.py)
TREC: 'CogComp/trec' failed (RuntimeError: Dataset scripts are no longer supported, but found trec.py)
TREC: NO candidate worked -- skipping this task.


In [66]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
import numpy as np

def find_text_label_columns(dataset_split):
    cols = dataset_split.column_names
    text_col = next((c for c in ["text", "sentence", "sentence1"] if c in cols), None)
    label_col = next((c for c in ["label", "label_text"] if c in cols), None)
    if text_col is None or label_col is None:
        raise ValueError(f"Couldn't identify text/label columns in {cols}")
    return text_col, label_col

def eval_classification_probe(encode_fn, dataset, task_name="", cv=5, max_samples=4000):
    split_name = "train" if "train" in dataset else list(dataset.keys())[0]
    split = dataset[split_name]
    if len(split) > max_samples:
        split = split.shuffle(seed=42).select(range(max_samples))

    text_col, label_col = find_text_label_columns(split)
    texts = list(split[text_col])
    labels = np.asarray(split[label_col])

    print(f"  [{task_name}] encoding {len(texts):,} examples...")
    embeddings = encode_fn(texts)

    clf = LogisticRegression(max_iter=1000)
    scores = cross_val_score(clf, embeddings, labels, cv=cv)
    mean_acc = float(scores.mean())
    print(f"  [{task_name}] {cv}-fold CV accuracy: {mean_acc:.4f} (+/- {scores.std():.4f})")
    return mean_acc

In [67]:
import pandas as pd, gc, torch

rows = []
for model_name, cfg in MODELS.items():
    print(f"\n=== {model_name} ===")
    if cfg["type"] == "st":
        encode_fn = make_st_encoder(cfg["id"], prefix=cfg.get("prefix", ""))
    elif cfg["type"] == "prebuilt":
        encode_fn = cfg["encode_fn"]

    row = {"model": model_name}
    for task_name, dataset in loaded_tasks.items():
        row[task_name] = eval_classification_probe(encode_fn, dataset, task_name=task_name)
    row["Avg"] = float(np.mean([v for k, v in row.items() if k != "model"]))
    rows.append(row)

probe_table = pd.DataFrame(rows).set_index("model")
display(probe_table.round(4))


=== Our model ===
  [MR] encoding 4,000 examples...
  [MR] 5-fold CV accuracy: 0.8580 (+/- 0.0094)
  [CR] encoding 3,394 examples...
  [CR] 5-fold CV accuracy: 0.9034 (+/- 0.0089)
  [SUBJ] encoding 4,000 examples...
  [SUBJ] 5-fold CV accuracy: 0.9303 (+/- 0.0064)
  [SST2] encoding 4,000 examples...
  [SST2] 5-fold CV accuracy: 0.9008 (+/- 0.0109)


,MR,CR,SUBJ,SST2,Avg
model,,,,,
Our model,0.858,0.9034,0.9302,0.9008,0.8981


In [69]:
# NOTE: this cell was moved UP from later in the notebook. It defines
# MODEL_ZOO (and the wrapper classes), which the evaluation cells below
# iterate over -- previously it sat AFTER its first real use, so a clean
# "Run All" raised NameError on the adversarial-retrieval cell.

from abc import ABC, abstractmethod

class EmbeddingModel(ABC):
    @abstractmethod
    def encode(self, sentences: List[str]) -> np.ndarray:
        """Return (N, D) float32 array of L2-normalised embeddings."""

class FunctionEmbeddingModel(EmbeddingModel):
    def __init__(self, fn, name="custom"):
        self._fn = fn
        self.name = name

    def encode(self, sentences: List[str]) -> np.ndarray:
        vecs = self._fn(list(sentences))
        norms = np.linalg.norm(vecs, axis=1, keepdims=True) + 1e-10
        return (vecs / norms).astype(np.float32)


class STWrapper(EmbeddingModel):
    """SentenceTransformer wrapper with optional query/passage prefixes."""

    def __init__(self, model_or_path, name, prefix="", device=None, st_model=None):
        self.name = name
        self.prefix = prefix
        self._st = st_model
        self._path = model_or_path
        self._device = device or str(DEVICE)

    @property
    def st(self):
        if self._st is None:
            self._st = SentenceTransformer(self._path, device=self._device)
            # Deliberately NOT overriding max_seq_length here. This used to force
        # every baseline to CFG.max_seq_length (128), but these models ship
        # with their own trained defaults (mpnet 384, BGE/E5/GTE 512), and
        # truncating them to 128 handicaps the baselines on any task with
        # longer inputs -- an unfair comparison against our model. Each model
        # now uses its own configured length.
            self._st.eval()
        return self._st

    def encode(self, sentences, batch_size=None, **kwargs):
        texts = [self.prefix + s for s in sentences] if self.prefix else list(sentences)
        return self.st.encode(
            texts,
            batch_size=batch_size or CFG.encode_batch_size,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        ).astype(np.float32)

    def free(self):
        self._st = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


def make_random_model(dim=768, seed=SEED):
    """Sanity check: should FAIL the criteria. If it passes, the eval is broken."""
    rng = np.random.default_rng(seed)
    def encode(sents):
        return rng.standard_normal((len(sents), dim)).astype(np.float32)
    return FunctionEmbeddingModel(encode, name="Random (sanity check)")

MODEL_ZOO = {
    "ours (align-mpnet)": lambda: STWrapper(
        str(merged_directory), name="ours (align-mpnet)"
    ),
    
}

print("Models in the comparison:")
for key in MODEL_ZOO:
    print(" -", key)

Models in the comparison:
 - ours (align-mpnet)


In [70]:
import numpy as np, pandas as pd, gc, torch
from sklearn.metrics import ndcg_score

def build_external_ranking_task(pairs_ds, pool_size=50, n_queries=1000, seed=42):
    """Relevance comes from the DATASET's own labels, not our taxonomy.
    For each anchor: rank its true paraphrase above `pool_size` distractors."""
    labels = np.asarray(pairs_ds["label"])
    pos_idx = np.where(labels == 1)[0]
    rng = np.random.RandomState(seed)
    query_idx = rng.choice(pos_idx, size=min(n_queries, len(pos_idx)), replace=False)

    s1, s2 = list(pairs_ds["sentence1"]), list(pairs_ds["sentence2"])
    neg_idx = np.where(labels == 0)[0]
    queries = []
    for i in query_idx:
        hard = neg_idx[neg_idx != i]
        n_hard = min(pool_size, len(hard))
        chosen = list(rng.choice(hard, size=n_hard, replace=False)) if n_hard else []
        if len(chosen) < pool_size:                     # top up with randoms
            taken = set(chosen) | {i}
            pool = [j for j in range(len(s2)) if j not in taken]
            chosen += list(rng.choice(pool, size=pool_size - len(chosen), replace=False))
        distractors = np.asarray(chosen)
        queries.append({"anchor_idx": int(i), "true_idx": int(i),
                        "distractor_idx": distractors.tolist()})
    return {"s1": s1, "s2": s2, "queries": queries}

def eval_external_ndcg(encode_fn, task, k=5, task_name=""):
    anchor_emb = encode_fn(task["s1"])
    cand_emb = encode_fn(task["s2"])
    ndcg_k, ndcg_full, mrrs = [], [], []
    for q in task["queries"]:
        a = anchor_emb[q["anchor_idx"]]
        idx = [q["true_idx"]] + q["distractor_idx"]
        scores = (cand_emb[idx] @ a).reshape(1, -1)
        relevance = np.array([[1] + [0] * len(q["distractor_idx"])], dtype=np.float64)
        ndcg_k.append(ndcg_score(relevance, scores, k=k))
        ndcg_full.append(ndcg_score(relevance, scores))
        rank = int((scores[0] > scores[0][0]).sum()) + 1
        mrrs.append(1.0 / rank)
    return {"ndcg_full": float(np.mean(ndcg_full)), f"ndcg@{k}": float(np.mean(ndcg_k)),
            "mrr": float(np.mean(mrrs)), "n_queries": len(ndcg_k)}

paws_rank_task = build_external_ranking_task(paws_test_ds, pool_size=50, n_queries=1000)
mrpc_rank_task = build_external_ranking_task(mrpc_test_ds, pool_size=50, n_queries=1000)
qqp_rank_task  = build_external_ranking_task(qqp_test_ds,  pool_size=50, n_queries=1000)
adv_rows = []
for label, factory in MODEL_ZOO.items():
    if label == "Random (sanity)":
        continue
    print(f"\n=== {label} ===")
    try:
        m_obj = factory()
    except Exception as error:
        print(f"  [skip] could not load: {error}")
        continue
    encode_fn = m_obj.encode

    p = eval_external_ndcg(encode_fn, paws_rank_task, k=5, task_name="PAWS")
    m = eval_external_ndcg(encode_fn, mrpc_rank_task, k=5, task_name="MRPC")
    q = eval_external_ndcg(encode_fn, qqp_rank_task,  k=5, task_name="QQP")

    adv_rows.append({"model": label,
                      "PAWS NDCG@5": p["ndcg@5"], "PAWS MRR": p["mrr"],
                      "MRPC NDCG@5": m["ndcg@5"], "MRPC MRR": m["mrr"],
                      "QQP NDCG@5": q["ndcg@5"],  "QQP MRR": q["mrr"]})
    print(f"PAWS={p['ndcg@5']:.4f}  MRPC={m['ndcg@5']:.4f}  QQP={q['ndcg@5']:.4f}")

    if hasattr(m_obj, "free"):
        m_obj.free()
    del m_obj, encode_fn
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

adv_df = pd.DataFrame(adv_rows).set_index("model")
display(adv_df.round(4))


=== ours (align-mpnet) ===
PAWS=0.9928  MRPC=1.0000  QQP=0.9996


,PAWS NDCG@5,PAWS MRR,MRPC NDCG@5,MRPC MRR,QQP NDCG@5,QQP MRR
model,,,,,,
ours (align-mpnet),0.9928,0.991,1.0,1.0,0.9996,0.9995


In [71]:
from dataclasses import field
from abc import ABC as _ABC

def cosine_sim(a, b):
    """Element-wise cosine similarity for two (N, D) arrays -> [-1, 1]."""
    return np.einsum("ij,ij->i", a, b).clip(-1, 1)

def ned(a, b):
    """Normalised Euclidean Distance in [0,1]: 0 = identical."""
    diff = np.linalg.norm(a - b, axis=1)
    denom = np.linalg.norm(a, axis=1) + np.linalg.norm(b, axis=1) + 1e-10
    return (diff / denom).clip(0, 1)

# ---- WordNet perturbations (module-level RNG, matching the original) ----
def get_synonyms(word, pos_tag):
    pos = _wn_pos(pos_tag)
    out = set()
    for ss in (wordnet.synsets(word, pos=pos) if pos else []):
        for lemma in ss.lemmas():
            s = lemma.name().replace("_", " ")
            if s.lower() != word.lower():
                out.add(s)
    return list(out)

def get_antonyms(word, pos_tag):
    pos = _wn_pos(pos_tag)
    out = set()
    for ss in (wordnet.synsets(word, pos=pos) if pos else []):
        for lemma in ss.lemmas():
            for ant in lemma.antonyms():
                out.add(ant.name().replace("_", " "))
    return list(out)

def eval_replace_words(sentence, n, mode="synonym"):
    tokens = nltk.word_tokenize(sentence)
    tags = nltk.pos_tag(tokens)
    candidates = [
        (i, w, t) for i, (w, t) in enumerate(tags)
        if t.startswith("J") or t.startswith("V")
    ]
    random.shuffle(candidates)
    replaced, count = tokens[:], 0
    for i, word, tag in candidates:
        if count >= n:
            break
        pool = get_synonyms(word, tag) if mode == "synonym" else get_antonyms(word, tag)
        if pool:
            replaced[i] = random.choice(pool)
            count += 1
    return " ".join(replaced) if count > 0 else None

def eval_jumble_sentence(sentence, n_swaps=3):
    tokens = nltk.word_tokenize(sentence)
    if len(tokens) < 4:
        return sentence
    for _ in range(min(n_swaps, len(tokens) // 2)):
        i, j = random.sample(range(len(tokens)), 2)
        tokens[i], tokens[j] = tokens[j], tokens[i]
    return " ".join(tokens)

# ---- Evaluation pair loaders (VALIDATION splits - never trained on) ----
def _eval_pairs(split, s1, s2, max_pairs):
    pos, neg = [], []
    labels = split["label"]
    for index in range(len(labels)):
        if len(pos) >= max_pairs and len(neg) >= max_pairs:
            break
        pair = (split[s1][index], split[s2][index])
        if labels[index] == 1 and len(pos) < max_pairs:
            pos.append(pair)
        elif labels[index] == 0 and len(neg) < max_pairs:
            neg.append(pair)
    return pos, neg

@dataclass
class CriterionResult:
    name: str
    condition_pct: float
    avg_margin: float
    details: dict = field(default_factory=dict)

    def __str__(self):
        mark = "PASS" if self.condition_pct > 0.5 else "FAIL"
        return (f"[{mark}] {self.name:<28}"
                f" condition={self.condition_pct:6.1%}"
                f" avg_margin={self.avg_margin:+.3f}")


@dataclass
class EvalReport:
    model_name: str
    eps: float
    results: List[CriterionResult]

    @property
    def n_satisfied(self):
        """Number of the four paper criteria (C1-C4) that are satisfied."""
        return sum(r.condition_pct > 0.5 for r in self.results)

    @property
    def mean_condition_pct(self):
        return float(np.mean([r.condition_pct for r in self.results]))

    def summary(self):
        lines = [
            "=" * 68,
            f"  AlignSim -- {self.model_name}  (eps={self.eps})",
            "=" * 68,
        ]
        for r in self.results:
            lines.append("  " + str(r))
            for k, v in r.details.items():
                lines.append(f"        {k}: {v}")
        lines.append("-" * 68)
        lines.append(f"  Criteria satisfied : {self.n_satisfied}/4")
        lines.append(f"  Mean condition_pct : {self.mean_condition_pct:.1%}")
        lines.append("=" * 68)
        return "\n".join(lines)


class EmbeddingEvaluator:
    def __init__(self, model, max_pairs=300, n_perturbations=2, eps=0.05,
                 relation_df=None, max_random_overlap=0.3,
                 seed=SEED, use_ned=False):
        self.model = model
        self.max_pairs = max_pairs
        self.n_pert = n_perturbations
        assert eps >= 0, (
            f"eps must be >= 0 (the paper defines it as an expected minimum "
            f"margin, eps >> 0); got {eps}."
        )
        self.eps = eps
        self.max_random_overlap = max_random_overlap
        self._related_index = self._build_related_index(relation_df)
        self.seed = seed
        self.use_ned = use_ned
        random.seed(seed)
        np.random.seed(seed)

    def _sim(self, a, b):
        return (1.0 - ned(a, b)) if self.use_ned else cosine_sim(a, b)

    def _embed(self, sentences):
        return self.model.encode(list(sentences))

    # Pairs known to be semantically EQUIVALENT, so random sampling never
    # accidentally draws one and calls it "unrelated". Populated from the
    # dataset's own relation labels (paraphrase / synonym / entailment).
    RELATED_RELATIONS = {"paraphrase", "synonym", "entailment"}

    def _build_related_index(self, dataframe):
        """Set of frozenset({a, b}) for every pair the DATA says is equivalent."""
        if dataframe is None:
            return set()
        related = dataframe[dataframe["relation_type"].isin(self.RELATED_RELATIONS)]
        return {frozenset((a, b)) for a, b in
                zip(related["anchor"], related["candidate"])}

    def _is_unrelated(self, s1, s2):
        """Reject self-pairs, known paraphrase/synonym/entailment pairs, and
        anything with high lexical overlap (a likely near-duplicate)."""
        if s1 == s2:
            return False
        if frozenset((s1, s2)) in self._related_index:
            return False
        return lexical_jaccard(s1, s2) <= self.max_random_overlap

    def _random_baseline(self, sentences, n=200):
        """alpha -- mean similarity of genuinely UNRELATED sentence pairs.

        Previously this paired two independent random samples with no checks,
        so a pair could be a true paraphrase (both sides of the same row are
        in the pool), a synonym variant, or even a sentence with itself.
        Any of those inflate alpha and make C2/C4 harder than intended.
        """
        pool = random.sample(sentences, min(len(sentences), 400))
        vecs = self._embed(pool)
        index = {s: i for i, s in enumerate(pool)}
        a_idx, b_idx, attempts = [], [], 0
        while len(a_idx) < n and attempts < n * 40:
            attempts += 1
            s1, s2 = random.choice(pool), random.choice(pool)
            if not self._is_unrelated(s1, s2):
                continue
            a_idx.append(index[s1]); b_idx.append(index[s2])
        if not a_idx:
            return 0.0
        return float(np.mean(self._sim(vecs[a_idx], vecs[b_idx])))

    # C1: Sim(S, S_P) - Sim(S, S_distinct) > eps
    # Uses REAL paraphrase/distinction rows from the custom dataset's quads
    # (test_df) as the paired positive/negative -- "distinction" is already
    # the dataset's own label for a genuinely unrelated candidate per
    # anchor, so no separate random-unrelated-pool sampling is needed.
    def criterion1(self, quads):
        if not quads:
            return CriterionResult("C1_semantic_distinction", 0.0, 0.0,
                                   {"error": "no distinction quads available"})
        s_orig = [q["original"] for q in quads]
        s_para = [q["paraphrase"] for q in quads]
        s_dist = [q["distinction"] for q in quads]
        sim_pos = self._sim(self._embed(s_orig), self._embed(s_para))
        sim_rnd = self._sim(self._embed(s_orig), self._embed(s_dist))
        margin = sim_pos - sim_rnd
        return CriterionResult(
            "C1_semantic_distinction",
            float(np.mean(margin > self.eps)), float(np.mean(margin)),
            {"condition": f"Sim(S,S_P) - Sim(S,S_distinct) > eps={self.eps}",
             "n_pairs": len(s_orig),
             "avg_paraphrase_sim": f"{np.mean(sim_pos):.3f}",
             "avg_distinct_sim": f"{np.mean(sim_rnd):.3f}"},
        )

    # C2: Sim(S, S_syn) - alpha > 0
    # Uses REAL synonym rows from the custom dataset's quads; alpha is the
    # mean similarity of the same anchors against their labeled
    # "distinction" candidate, instead of a randomly sampled unrelated-pair
    # pool.
    def criterion2(self, quads):
        if not quads:
            return CriterionResult("C2_synonym_replacement", 0.0, 0.0,
                                   {"error": "no synonym quads available"})
        s_orig = [q["original"] for q in quads]
        s_syn = [q["synonym"] for q in quads]
        s_dist = [q["distinction"] for q in quads]
        sims = self._sim(self._embed(s_orig), self._embed(s_syn))
        alpha = float(np.mean(self._sim(self._embed(s_orig), self._embed(s_dist))))
        margin = sims - alpha
        return CriterionResult(
            "C2_synonym_replacement",
            float(np.mean(margin > 0)), float(np.mean(margin)),
            {"condition": "Sim(S, S_syn) - alpha > 0", "n_pairs": len(s_orig),
             "avg_sim": f"{np.mean(sims):.3f}", "random_alpha": f"{alpha:.3f}"},
        )

    # C3: Sim(S, S_P) - Sim(S, S_ant) > eps
    # Uses REAL antonym rows from the custom dataset's quads (test_df),
    # not eval_replace_words() WordNet substitution.
    def criterion3(self, quads):
        if not quads:
            return CriterionResult("C3_antonym_replacement", 0.0, 0.0,
                                   {"error": "no antonym quads available"})
        s_orig = [q["original"] for q in quads]
        s_para = [q["paraphrase"] for q in quads]
        s_ant = [q["antonym"] for q in quads]
        sim_para = self._sim(self._embed(s_orig), self._embed(s_para))
        sim_ant = self._sim(self._embed(s_orig), self._embed(s_ant))
        margin = sim_para - sim_ant
        return CriterionResult(
            "C3_antonym_replacement",
            float(np.mean(margin > self.eps)), float(np.mean(margin)),
            {"condition": f"Sim(S,S_P) - Sim(S,S_ant) > eps={self.eps}",
             "n_pairs": len(s_orig),
             "avg_para_sim": f"{np.mean(sim_para):.3f}",
             "avg_antonym_sim": f"{np.mean(sim_ant):.3f}"},
        )

    # C4 (paper Criterion-4): Sim(S, S_P) - Sim(S, S_jum) > eps
    # Uses REAL jumbled rows from the custom dataset's quads (test_df),
    # not eval_jumble_sentence() on-the-fly shuffling.
    def criterion4(self, quads):
        if not quads:
            return CriterionResult("C4_sentence_jumbling", 0.0, 0.0,
                                   {"error": "no jumbled quads available"})
        s_orig = [q["original"] for q in quads]
        s_para = [q["paraphrase"] for q in quads]
        s_jum = [q["jumbled"] for q in quads]
        sim_para = self._sim(self._embed(s_orig), self._embed(s_para))
        sim_jum = self._sim(self._embed(s_orig), self._embed(s_jum))
        margin = sim_para - sim_jum
        return CriterionResult(
            "C4_sentence_jumbling",
            float(np.mean(margin > self.eps)), float(np.mean(margin)),
            {"condition": f"Sim(S,S_P) - Sim(S,S_jum) > eps={self.eps}",
             "n_pairs": len(s_orig),
             "avg_para_sim": f"{np.mean(sim_para):.3f}",
             "avg_jumble_sim": f"{np.mean(sim_jum):.3f}"},
        )

    def evaluate(self, shared=None):
        if shared is None:
            shared = build_alignsim_pairs(self.max_pairs)
        results = [
            # --- the paper's four criteria, all sourced from the custom
            # dataset's quads now ---
            self.criterion1(shared["quads"]),
            self.criterion2(shared["quads"]),
            self.criterion3(shared["quads"]),
            self.criterion4(shared["quads"]),
        ]
        name = getattr(self.model, "name", type(self.model).__name__)
        return EvalReport(name, self.eps, results)


def build_alignsim_pairs(max_pairs=300, seed=SEED):
    """Build the AlignSim quads ONCE and reuse across all models, so every
    model is scored on identical inputs.

    All four criteria (C1-C4) now draw from the custom dataset's held-out
    test split (test_df: anchor, candidate, relation_type) -- real
    paraphrase/synonym/antonym/jumbled/distinction rows per anchor --
    instead of QQP/MRPC/PAWS validation and WordNet-generated
    perturbations.
    """
    random.seed(seed)
    np.random.seed(seed)
    quad_items = build_relation_quads(
        test_df,
        required=("paraphrase", "synonym", "antonym", "jumbled", "distinction"),
        max_quads=max_pairs, seed=seed,
    )
    quads = [
        {"original": anchor, "paraphrase": row["paraphrase"],
         "synonym": row["synonym"], "antonym": row["antonym"],
         "jumbled": row["jumbled"], "distinction": row["distinction"]}
        for anchor, row in quad_items
    ]
    print(f"AlignSim quads (from custom dataset, test_df): {len(quads):,} anchors "
          f"with paraphrase/synonym/antonym/jumbled/distinction (C1-C4 all real)")
    return {"quads": quads}

print("AlignSim evaluator ready.")

AlignSim evaluator ready.


In [72]:
ALIGNSIM_EPS_VALUES = [0.05, 0.10, 0.20, 0.30]
ALIGNSIM_PAIRS = 300

shared_pairs = build_alignsim_pairs(ALIGNSIM_PAIRS)

criteria_cols = [
    "C1_semantic_distinction", "C2_synonym_replacement", "C3_antonym_replacement",
    "C4_sentence_jumbling",
]

alignsim_reports_by_eps = {eps: {} for eps in ALIGNSIM_EPS_VALUES}
alignsim_rows_by_eps = {eps: [] for eps in ALIGNSIM_EPS_VALUES}

for label, factory in MODEL_ZOO.items():
    print(f"\n--- AlignSim: {label} ---")
    try:
        m = factory()
    except Exception as error:
        print(f"  [skip] could not load: {error}")
        continue

    for eps in ALIGNSIM_EPS_VALUES:
        report = EmbeddingEvaluator(
            m, max_pairs=ALIGNSIM_PAIRS, n_perturbations=2,
            eps=eps, seed=SEED,
            relation_df=test_df,   # vestigial: only feeds _is_unrelated/
                                   # _random_baseline, which C1/C2 no longer
                                   # call now that they use real distinction/
                                   # synonym rows from the custom dataset
        ).evaluate(shared=shared_pairs)

        print(f"  eps={eps:.2f}: {report.summary()}")
        alignsim_reports_by_eps[eps][label] = report

        row = {"model": label}
        for r in report.results:
            row[r.name] = r.condition_pct * 100
            row[r.name + "_margin"] = r.avg_margin
        row["criteria_satisfied"] = report.n_satisfied
        row["mean_condition_pct"] = report.mean_condition_pct * 100
        alignsim_rows_by_eps[eps].append(row)

    if hasattr(m, "free"):
        m.free()
    del m
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

margin_cols = [c + "_margin" for c in criteria_cols]
alignsim_tables_by_eps = {}
alignsim_margins_by_eps = {}

for eps in ALIGNSIM_EPS_VALUES:
    table = pd.DataFrame(alignsim_rows_by_eps[eps])[
        ["model"] + criteria_cols + ["criteria_satisfied", "mean_condition_pct"]
    ]
    print(f"\n\ncondition_pct (%) @ eps={eps:.2f} -- higher is better; >50% = criterion satisfied")
    display(table.round(1))
    table.to_csv(Path(CFG.output_dir) / f"alignsim_comparison_eps{eps:.2f}.csv", index=False)
    alignsim_tables_by_eps[eps] = table

    margins = pd.DataFrame(alignsim_rows_by_eps[eps])[["model"] + margin_cols]
    print(f"\navg_margin @ eps={eps:.2f} -- positive = correct direction")
    display(margins.round(3))
    margins.to_csv(Path(CFG.output_dir) / f"alignsim_margins_eps{eps:.2f}.csv", index=False)
    alignsim_margins_by_eps[eps] = margins

    # Sanity: the random model MUST fail. If it passes, the evaluator is broken.
    rnd_reports = alignsim_reports_by_eps[eps]
    if "Random (sanity)" in rnd_reports:
        rnd = rnd_reports["Random (sanity)"]
        print(f"sanity check @ eps={eps:.2f} -- random model satisfied {rnd.n_satisfied}/4 "
              f"criteria (expected 0-1; if high, the evaluator is broken)")
ALIGNSIM_EPS = CFG.criteria_eps
alignsim_reports = alignsim_reports_by_eps[ALIGNSIM_EPS]
alignsim_table = alignsim_tables_by_eps[ALIGNSIM_EPS]
alignsim_margins = alignsim_margins_by_eps[ALIGNSIM_EPS]


AlignSim quads (from custom dataset, test_df): 300 anchors with paraphrase/synonym/antonym/jumbled/distinction (C1-C4 all real)

--- AlignSim: ours (align-mpnet) ---
  eps=0.05: ====================================================================
  AlignSim -- ours (align-mpnet)  (eps=0.05)
  [PASS] C1_semantic_distinction      condition=100.0% avg_margin=+0.738
        condition: Sim(S,S_P) - Sim(S,S_distinct) > eps=0.05
        n_pairs: 300
        avg_paraphrase_sim: 0.932
        avg_distinct_sim: 0.193
  [PASS] C2_synonym_replacement       condition=100.0% avg_margin=+0.787
        condition: Sim(S, S_syn) - alpha > 0
        n_pairs: 300
        avg_sim: 0.981
        random_alpha: 0.193
  [PASS] C3_antonym_replacement       condition= 68.3% avg_margin=+0.105
        condition: Sim(S,S_P) - Sim(S,S_ant) > eps=0.05
        n_pairs: 300
        avg_para_sim: 0.932
        avg_antonym_sim: 0.827
  [PASS] C4_sentence_jumbling         condition= 92.0% avg_margin=+0.353
        conditi

,model,C1_semantic_distinction,C2_synonym_replacement,C3_antonym_replacement,C4_sentence_jumbling,criteria_satisfied,mean_condition_pct
0,ours (align-mpnet),100.0,100.0,68.3,92.0,4,90.1



avg_margin @ eps=0.05 -- positive = correct direction


,model,C1_semantic_distinction_margin,C2_synonym_replacement_margin,C3_antonym_replacement_margin,C4_sentence_jumbling_margin
0,ours (align-mpnet),0.738,0.787,0.105,0.353




condition_pct (%) @ eps=0.10 -- higher is better; >50% = criterion satisfied


...

,model,C1_semantic_distinction_margin,C2_synonym_replacement_margin,C3_antonym_replacement_margin,C4_sentence_jumbling_margin
0,ours (align-mpnet),0.738,0.787,0.105,0.353




condition_pct (%) @ eps=0.30 -- higher is better; >50% = criterion satisfied


,model,C1_semantic_distinction,C2_synonym_replacement,C3_antonym_replacement,C4_sentence_jumbling,criteria_satisfied,mean_condition_pct
0,ours (align-mpnet),100.0,100.0,2.7,64.3,3,66.8



avg_margin @ eps=0.30 -- positive = correct direction


,model,C1_semantic_distinction_margin,C2_synonym_replacement_margin,C3_antonym_replacement_margin,C4_sentence_jumbling_margin
0,ours (align-mpnet),0.738,0.787,0.105,0.353


In [73]:
!pip -q install -U "mteb==1.39.7"
import mteb, json, shutil, gc, torch, numpy as np, pandas as pd
from pathlib import Path
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModel
import torch.nn.functional as F

MTEB_TASKS = {
    "similarity":         ["STSBenchmark", "SICK-R"],
    "pair_classification": ["SprintDuplicateQuestions", "TwitterSemEval2015",
                            "TwitterURLCorpus"],
    # HotpotQA / NQ / MSMARCO are FULL-CORPUS BEIR retrieval tasks -- corpora
    # run from ~100k (NQ) to several million passages (MSMARCO, HotpotQA).
    # Embedding + indexing the corpus (not the query count) is what makes
    # these slow, so they're cut here to keep eval time reasonable.
    # SciFact/NFCorpus/ArguAna are much smaller BEIR corpora and stay in.
    "retrieval":           ["SciFact", "NFCorpus", "ArguAna"],
    "reranking":           ["SummEval", "StackOverflowDupQuestions", "SciDocsRR"],
    "classification":      ["Banking77Classification", "AmazonCounterfactualClassification"],
    "clustering":          ["TwentyNewsgroupsClustering", "RedditClustering"],
}
METRIC = {
    "STSBenchmark": "Spearman", "SICK-R": "Spearman",
    "SprintDuplicateQuestions": "AP", "TwitterSemEval2015": "AP", "TwitterURLCorpus": "AP",
    "SciFact": "nDCG@10", "NFCorpus": "nDCG@10", "ArguAna": "nDCG@10",
    "SummEval": "Spearman", "StackOverflowDupQuestions": "MAP", "SciDocsRR": "MAP",
    "Banking77Classification": "Accuracy", "AmazonCounterfactualClassification": "Accuracy",
    "TwentyNewsgroupsClustering": "V-measure", "RedditClustering": "V-measure",
}
ALL_TASKS = [t for g in MTEB_TASKS.values() for t in g]

In [74]:
MTEB_TASKS = {
    "similarity":         ["STSBenchmark", "SICK-R"],
    "pair_classification": ["SprintDuplicateQuestions", "TwitterSemEval2015",
                            "TwitterURLCorpus"],
    # HotpotQA / NQ / MSMARCO are FULL-CORPUS BEIR retrieval tasks -- corpora
    # run from ~100k (NQ) to several million passages (MSMARCO, HotpotQA).
    # Embedding + indexing the corpus (not the query count) is what makes
    # these slow, so they're cut here to keep eval time reasonable.
    # SciFact/NFCorpus/ArguAna are much smaller BEIR corpora and stay in.
    "retrieval":           ["SciFact", "NFCorpus", "ArguAna"],
    "reranking":           ["SummEval", "StackOverflowDupQuestions", "SciDocsRR"],
    "classification":      ["Banking77Classification", "AmazonCounterfactualClassification"],
    "clustering":          ["TwentyNewsgroupsClustering", "RedditClustering"],
}
METRIC = {
    "STSBenchmark": "Spearman", "SICK-R": "Spearman",
    "SprintDuplicateQuestions": "AP", "TwitterSemEval2015": "AP", "TwitterURLCorpus": "AP",
    "SciFact": "nDCG@10", "NFCorpus": "nDCG@10", "ArguAna": "nDCG@10",
    "SummEval": "Spearman", "StackOverflowDupQuestions": "MAP", "SciDocsRR": "MAP",
    "Banking77Classification": "Accuracy", "AmazonCounterfactualClassification": "Accuracy",
    "TwentyNewsgroupsClustering": "V-measure", "RedditClustering": "V-measure",
}
ALL_TASKS = [t for g in MTEB_TASKS.values() for t in g]

In [75]:
MTEB_ROOT = Path(CFG.output_dir) / "mteb"
FORCE_MTEB_RECOMPUTE = True     # True on first run: old cached results are from
                                # the 128-token model and would come back stale
if FORCE_MTEB_RECOMPUTE and MTEB_ROOT.exists():
    shutil.rmtree(MTEB_ROOT)
MTEB_ROOT.mkdir(parents=True, exist_ok=True)

def _as_text(x):
    """MTEB corpus rows arrive as {title, text} dicts, not strings."""
    return (x.get("title","") + " " + x.get("text","")).strip() if isinstance(x, dict) else str(x)

class MTEBModel:
    """Wraps an encoder for MTEB. E5 needs 'query: ' on queries and 'passage: '
    on documents for retrieval; applying 'query: ' to the corpus understates it."""
    def __init__(self, st, q_prefix="", d_prefix=None):
        self.st, self.q, self.d = st, q_prefix, (q_prefix if d_prefix is None else d_prefix)
    def _enc(self, xs, pfx, bs):
        return self.st.encode([pfx + _as_text(x) for x in xs] if pfx else [_as_text(x) for x in xs],
                              batch_size=bs, convert_to_numpy=True,
                              normalize_embeddings=True, show_progress_bar=False)
    def encode(self, s, batch_size=128, **kw):         return self._enc(s, self.q, batch_size)
    def encode_queries(self, s, batch_size=128, **kw): return self._enc(s, self.q, batch_size)
    def encode_corpus(self, s, batch_size=128, **kw):  return self._enc(s, self.d, batch_size)

def build_for_mteb(name):
    """MODEL_ZOO values are zero-arg FACTORIES returning a wrapper with .encode(),
    not config dicts -- calling cfg["type"] on a lambda is what raised
    TypeError: 'function' object is not subscriptable.

    We call the factory, then add the query/corpus methods MTEB wants for
    retrieval. E5 must use 'passage: ' on documents; applying its 'query: '
    prefix to the corpus understates it badly on SciFact/NFCorpus/ArguAna.
    """
    inner = MODEL_ZOO[name]()                      # <-- call it

    st  = getattr(inner, "st", None)               # SentenceTransformer, if any
    qpx = getattr(inner, "prefix", "") or ""
    dpx = "passage: " if qpx.strip() == "query:" else qpx

    class _Wrap:
        def _enc(self, xs, pfx, bs):
            texts = [_as_text(x) for x in xs]
            if st is not None:                     # bypass inner.encode so the
                if pfx: texts = [pfx + t for t in texts]   # corpus prefix applies
                return st.encode(texts, batch_size=bs, convert_to_numpy=True,
                                 normalize_embeddings=True, show_progress_bar=False)
            v = np.asarray(inner.encode(texts, batch_size=bs))   # non-ST wrappers (e.g. random sanity model)
            return v / np.linalg.norm(v, axis=1, keepdims=True).clip(min=1e-9)
        def encode(self, s, batch_size=128, **kw):         return self._enc(s, qpx, batch_size)
        def encode_queries(self, s, batch_size=128, **kw): return self._enc(s, qpx, batch_size)
        def encode_corpus(self, s, batch_size=128, **kw):  return self._enc(s, dpx, batch_size)
    w = _Wrap(); w._inner = inner                  # keep a ref so .free() works
    return w

In [76]:
MTEB_ROOT = Path(CFG.output_dir) / "mteb"
MTEB_ROOT.mkdir(parents=True, exist_ok=True)

# IMPORTANT:
# Do NOT automatically delete MTEB_ROOT here.
#
# If you need to remove the OLD 128-token results, run these TWO lines
# manually ONCE in a separate cell:
#
# shutil.rmtree(MTEB_ROOT, ignore_errors=True)
# MTEB_ROOT.mkdir(parents=True, exist_ok=True)


def run_mteb(model, tasks, folder):
    """Run the requested MTEB tasks once."""
    
    task_objs = mteb.get_tasks(tasks=tasks)

    evaluator = mteb.MTEB(tasks=task_objs)

    return evaluator.run(
        model,
        output_folder=folder,
        eval_splits=["test"],
        verbosity=0,
    )


def completed(folder):
    done = set()

    for p in Path(folder).rglob("*.json"):

        if p.name in {"model_meta.json", "model_metadata.json"}:
            continue

        try:
            with open(p, "r", encoding="utf-8") as f:
                d = json.load(f)
        except Exception:
            continue

        if not isinstance(d, dict):
            continue

        n = d.get("task_name") or p.stem

        if n in ALL_TASKS and d.get("scores"):
            done.add(n)

    return done


# ============================================================
# RUN MTEB
# ============================================================

for name in MODEL_ZOO:

    folder = MTEB_ROOT / name.replace("/", "_").replace(" ", "_")
    folder.mkdir(parents=True, exist_ok=True)

    done = completed(folder)
    todo = [t for t in ALL_TASKS if t not in done]

    print(f"\n{'=' * 70}")
    print(name)
    print(f"completed : {len(done)}/{len(ALL_TASKS)}")
    print(f"remaining : {len(todo)}/{len(ALL_TASKS)}")
    print(f"todo      : {todo}")
    print("=" * 70)

    if not todo:
        print(f"[cached] {name}")
        continue

    m = None

    try:
        m = build_for_mteb(name)

        # --------------------------------------------------
        # Run ONE TASK AT A TIME
        # --------------------------------------------------
        for i, task in enumerate(todo, 1):

            # Check again in case this cell was interrupted
            if task in completed(folder):
                print(f"[cached] {task}")
                continue

            print(
                f"\n[{i}/{len(todo)}] "
                f"{name} -> {task}"
            )

            try:
                run_mteb(
                    m,
                    [task],
                    str(folder)
                )

                # Verify that MTEB actually wrote the JSON
                new_done = completed(folder)

                if task in new_done:
                    print(f"  [done] {task}")
                else:
                    print(f"  [warning] {task} finished but no result JSON detected")

            except Exception as e:
                print(
                    f"  [FAILED] {task}: "
                    f"{type(e).__name__}: {e}"
                )

                # Do NOT stop the other tasks
                continue

    except Exception as e:
        print(
            f"[MODEL ERROR] {name}: "
            f"{type(e).__name__}: {e}"
        )

    finally:
        if m is not None:
            del m

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


# ============================================================
# FINAL COMPLETION REPORT
# ============================================================

print("\n\nFINAL COMPLETION")
print("=" * 70)

for name in MODEL_ZOO:

    folder = MTEB_ROOT / name.replace("/", "_").replace(" ", "_")

    done = completed(folder) if folder.exists() else set()
    missing = [t for t in ALL_TASKS if t not in done]

    status = "COMPLETE" if len(done) == len(ALL_TASKS) else "INCOMPLETE"

    print(
        f"{name:<25} "
        f"{len(done)}/{len(ALL_TASKS):<3} "
        f"{status}"
    )

    if missing:
        print("   missing:", ", ".join(missing))


ours (align-mpnet)
completed : 0/15
remaining : 15/15
todo      : ['STSBenchmark', 'SICK-R', 'SprintDuplicateQuestions', 'TwitterSemEval2015', 'TwitterURLCorpus', 'SciFact', 'NFCorpus', 'ArguAna', 'SummEval', 'StackOverflowDupQuestions', 'SciDocsRR', 'Banking77Classification', 'AmazonCounterfactualClassification', 'TwentyNewsgroupsClustering', 'RedditClustering']

[1/15] ours (align-mpnet) -> STSBenchmark


  [done] STSBenchmark

[2/15] ours (align-mpnet) -> SICK-R


  [done] SICK-R

[3/15] ours (align-mpnet) -> SprintDuplicateQuestions


  [done] SprintDuplicateQuestions

[4/15] ours (align-mpnet) -> TwitterSemEval2015


  [done] TwitterSemEval2015

[5/15] ours (align-mpnet) -> TwitterURLCorpus


...

Clustering: 100%|██████████| 10/10 [01:07<00:00,  6.72s/it]


  [done] TwentyNewsgroupsClustering

[15/15] ours (align-mpnet) -> RedditClustering


test.jsonl:   0%|          | 0.00/35.9M [00:00<?, ?B/s]

Clustering: 100%|██████████| 25/25 [17:10<00:00, 41.20s/it]


  [done] RedditClustering


FINAL COMPLETION
ours (align-mpnet)        14/15  INCOMPLETE
   missing: AmazonCounterfactualClassification


In [77]:
print("ALL_TASKS:")
for t in ALL_TASKS:
    print(repr(t), type(t))

folder = MTEB_ROOT / "ours_(align-mpnet)"

print("\nDetected JSON tasks:")
for p in folder.rglob("*.json"):
    if p.name in {"model_meta.json", "model_metadata.json"}:
        continue

    d = json.load(open(p))
    print(
        repr(d.get("task_name")),
        "in ALL_TASKS =", d.get("task_name") in ALL_TASKS,
        "scores truthy =", bool(d.get("scores"))
    )

ALL_TASKS:
'STSBenchmark' <class 'str'>
'SICK-R' <class 'str'>
'SprintDuplicateQuestions' <class 'str'>
'TwitterSemEval2015' <class 'str'>
'TwitterURLCorpus' <class 'str'>
'SciFact' <class 'str'>
'NFCorpus' <class 'str'>
'ArguAna' <class 'str'>
'SummEval' <class 'str'>
'StackOverflowDupQuestions' <class 'str'>
'SciDocsRR' <class 'str'>
'Banking77Classification' <class 'str'>
'AmazonCounterfactualClassification' <class 'str'>
'TwentyNewsgroupsClustering' <class 'str'>
'RedditClustering' <class 'str'>

Detected JSON tasks:
'TwentyNewsgroupsClustering' in ALL_TASKS = True scores truthy = True
'TwitterSemEval2015' in ALL_TASKS = True scores truthy = True
'RedditClustering' in ALL_TASKS = True scores truthy = True
'StackOverflowDupQuestions' in ALL_TASKS = True scores truthy = True
'STSBenchmark' in ALL_TASKS = True scores truthy = True
'Banking77Classification' in ALL_TASKS = True scores truthy = True
'SciFact' in ALL_TASKS = True scores truthy = True
'SummEval' in ALL_TASKS = True scores 

In [78]:
def scores_in(folder):
    out = {}
    for p in Path(folder).rglob("*.json"):
        if p.name in {"model_meta.json","model_metadata.json"}: continue
        try: d = json.load(open(p))
        except Exception: continue
        if not isinstance(d, dict) or "scores" not in d: continue
        t = d["scores"].get("test")
        if not t: continue
        e = t[0] if isinstance(t, list) else t
        v = e.get("main_score") or e.get("cos_sim",{}).get("ap") or e.get("cos_sim",{}).get("spearman")
        if v is not None: out[d.get("task_name") or p.stem] = v*100
    return out

scores = {f.name.replace("_"," "): scores_in(f)
          for f in sorted(MTEB_ROOT.iterdir()) if f.is_dir() and scores_in(f)}
mteb_scores = pd.DataFrame(scores).T
# Matches "MPNet-base (baseline)" (or any future row we label "baseline").
base = next((i for i in mteb_scores.index if "baseline" in i.lower()), None)

for group, tasks in MTEB_TASKS.items():
    have = [t for t in tasks if t in mteb_scores.columns]
    if not have: print(f"[{group}] not run"); continue
    v = mteb_scores[have].copy()
    v.columns = [f"{t} ({METRIC[t]})" for t in have]
    v[f"MEAN_{group}"] = mteb_scores[have].mean(axis=1)
    v = v.sort_values(f"MEAN_{group}", ascending=False)
    print(f"\n===== {group} =====");  display(v.round(2))
    if base:
        print(f"delta vs {base}");   display(v.subtract(v.loc[base], axis=1).round(2))
    v.to_csv(Path(CFG.output_dir)/f"mteb_{group}.csv")


===== similarity =====


,STSBenchmark (Spearman),SICK-R (Spearman),MEAN_similarity
ours (align-mpnet),85.0,76.97,80.99



===== pair_classification =====


,SprintDuplicateQuestions (AP),TwitterSemEval2015 (AP),TwitterURLCorpus (AP),MEAN_pair_classification
ours (align-mpnet),89.46,74.56,85.72,83.25



===== retrieval =====


...

,SummEval (Spearman),StackOverflowDupQuestions (MAP),SciDocsRR (MAP),MEAN_reranking
ours (align-mpnet),28.07,51.69,88.18,55.98



===== classification =====


,Banking77Classification (Accuracy),MEAN_classification
ours (align-mpnet),80.58,80.58



===== clustering =====


,TwentyNewsgroupsClustering (V-measure),RedditClustering (V-measure),MEAN_clustering
ours (align-mpnet),50.22,53.29,51.76


## Retrieval detail: nDCG@10, Recall@K, MRR@10 (SciFact / NFCorpus / ArguAna)

`scores_in()` above only pulls each task's single `main_score` (nDCG@10 for retrieval), which is fine for the group summary table but hides Recall@K and MRR@10. This cell reads the raw MTEB result JSON directly to pull out whichever `ndcg_at_*`, `recall_at_*`, and `mrr_at_*` keys are actually present, for the retrieval tasks specifically.

In [79]:
def retrieval_detail_in(folder, tasks):
    """Pull ndcg_at_k / recall_at_k / mrr_at_k straight out of the raw MTEB
    result JSON for the given retrieval task names. Returns
    {task_name: {metric_key: value}}."""
    out = {}
    for p in Path(folder).rglob("*.json"):
        if p.name in {"model_meta.json", "model_metadata.json"}:
            continue
        try:
            d = json.load(open(p))
        except Exception:
            continue
        if not isinstance(d, dict) or "scores" not in d:
            continue
        task_name = d.get("task_name") or p.stem
        if task_name not in tasks:
            continue
        test_scores = d["scores"].get("test")
        if not test_scores:
            continue
        entry = test_scores[0] if isinstance(test_scores, list) else test_scores
        metrics = {
            k: v * 100 for k, v in entry.items()
            if isinstance(v, (int, float)) and (
                k.startswith("ndcg_at_") or k.startswith("recall_at_") or k.startswith("mrr_at_")
            )
        }
        out[task_name] = metrics
    return out

RETRIEVAL_TASKS = MTEB_TASKS.get("retrieval", [])
retrieval_detail_rows = {}
for f in sorted(MTEB_ROOT.iterdir()):
    if not f.is_dir():
        continue
    model_name = f.name.replace("_", " ")
    detail = retrieval_detail_in(f, RETRIEVAL_TASKS)
    if detail:
        retrieval_detail_rows[model_name] = detail

for task in RETRIEVAL_TASKS:
    rows = {
        model_name: detail[task]
        for model_name, detail in retrieval_detail_rows.items()
        if task in detail
    }
    if not rows:
        continue
    task_df = pd.DataFrame(rows).T
    # keep a readable, consistent column order when present
    preferred_cols = [c for c in [
        "ndcg_at_10", "ndcg_at_100",
        "recall_at_10", "recall_at_100",
        "mrr_at_10", "mrr_at_100",
    ] if c in task_df.columns]
    other_cols = [c for c in task_df.columns if c not in preferred_cols]
    task_df = task_df[preferred_cols + other_cols]
    print(f"\n===== {task}: nDCG@10 / Recall@K / MRR@10 =====")
    display(task_df.round(2))
    task_df.to_csv(Path(CFG.output_dir) / f"mteb_retrieval_detail_{task}.csv")



===== SciFact: nDCG@10 / Recall@K / MRR@10 =====


,ndcg_at_10,ndcg_at_100,recall_at_10,recall_at_100,mrr_at_10,mrr_at_100,ndcg_at_1,ndcg_at_3,ndcg_at_5,ndcg_at_20,...,recall_at_1,recall_at_3,recall_at_5,recall_at_20,recall_at_1000,mrr_at_1,mrr_at_3,mrr_at_5,mrr_at_20,mrr_at_1000
ours (align-mpnet),63.59,66.99,78.42,94.67,59.49,60.03,50.0,57.86,60.91,64.93,...,47.97,63.12,70.63,83.57,99.67,50.0,56.94,58.54,59.77,60.05



===== NFCorpus: nDCG@10 / Recall@K / MRR@10 =====


,ndcg_at_10,ndcg_at_100,recall_at_10,recall_at_100,mrr_at_10,mrr_at_100,ndcg_at_1,ndcg_at_3,ndcg_at_5,ndcg_at_20,...,recall_at_1,recall_at_3,recall_at_5,recall_at_20,recall_at_1000,mrr_at_1,mrr_at_3,mrr_at_5,mrr_at_20,mrr_at_1000
ours (align-mpnet),31.27,29.66,15.23,31.51,50.08,50.73,38.54,35.79,33.85,29.73,...,4.66,8.74,11.28,19.93,63.48,40.87,47.88,49.25,50.42,50.78



===== ArguAna: nDCG@10 / Recall@K / MRR@10 =====


,ndcg_at_10,ndcg_at_100,recall_at_10,recall_at_100,mrr_at_10,mrr_at_100,ndcg_at_1,ndcg_at_3,ndcg_at_5,ndcg_at_20,...,recall_at_1,recall_at_3,recall_at_5,recall_at_20,recall_at_1000,mrr_at_1,mrr_at_3,mrr_at_5,mrr_at_20,mrr_at_1000
ours (align-mpnet),48.6,53.35,78.59,98.86,39.47,40.63,24.18,37.22,42.48,51.94,...,24.18,46.8,59.6,91.61,99.64,24.47,34.07,36.91,40.41,40.63
